# ReqCheck: historical CPU research notebook
**6-credit BSSE FYP · technology cutoff: before 1 April 2020**

Run cells in order in a Colab **CPU** runtime. This self-contained notebook includes the research source, historical requirements, original PROMISE-format dataset, cleaned research CSV, provenance/license notices and a synthetic smoke fixture. No GPU/TPU is needed.

The current Colab host is not a 2020 operating system. An isolated Python 3.7.6 interpreter runs all experiments with pinned pre-cutoff libraries; host cells provision files, invoke that interpreter, display images and download outputs. The vendor Miniconda py37_4.8.2 installer is checksum-verified.

The bundled benchmark was retrieved from a current mirror carrying a 2007 copyright and CC BY-SA 3.0 notice. Historical byte identity is not independently established. Two conflicting-label rows and one cross-project near duplicate were excluded; consistent exact duplicates are removed by the runner. Review data/provenance.json and data/DATA_LICENSE.md. These are exploratory retrospective experiments, not a backdated 2020 study.

The historical environment and full binary/NFR code paths were exercised on Linux x86-64 when the package was created. Hosted Colab execution still depends on current service/network availability. Demo-fixture scores are never research evidence.


In [ ]:
import pathlib, platform, urllib.request, hashlib, subprocess, os, ssl, json
BASE = pathlib.Path('/content/reqcheck_2020') if pathlib.Path('/content').exists() else pathlib.Path.cwd() / 'reqcheck_2020_runtime'
BASE.mkdir(parents=True, exist_ok=True)
if platform.system() != 'Linux' or platform.machine() not in ('x86_64', 'AMD64'):
    raise RuntimeError('This bootstrap requires Linux x86-64; use a Colab CPU runtime.')
PREFIX = BASE / 'legacy_python'
PY = PREFIX / 'bin/python'
INSTALLER = BASE / 'Miniconda3-py37_4.8.2-Linux-x86_64.sh'
URL = 'https://repo.anaconda.com/miniconda/' + INSTALLER.name
EXPECTED = '957d2f0f0701c3d1335e3b39f235d197837ad69a944fa6f5d8ad2c686b69df3b'
if not PY.exists():
    if not INSTALLER.exists():
        urllib.request.urlretrieve(URL, str(INSTALLER))
    if hashlib.sha256(INSTALLER.read_bytes()).hexdigest() != EXPECTED:
        raise RuntimeError('Installer checksum mismatch; stop and investigate.')
    subprocess.run(['bash', str(INSTALLER), '-b', '-p', str(PREFIX)], check=True)
ENV = os.environ.copy()
ENV['OMP_NUM_THREADS'] = '1'
ENV['OPENBLAS_NUM_THREADS'] = '1'
# Use the host trust store if available; never disable TLS verification.
ca = ssl.get_default_verify_paths().cafile
if ca:
    ENV['PIP_CERT'] = ca
    ENV['SSL_CERT_FILE'] = ca
subprocess.run([str(PY), '-V'], check=True, env=ENV)


In [ ]:
FILES = {'config/requirements-bootstrap-2020.txt': 'pip==20.0.2\nsetuptools==46.0.0\nwheel==0.34.2\n', 'config/requirements-research-2020.txt': 'numpy==1.18.2\nscipy==1.4.1\npandas==1.0.3\nscikit-learn==0.22.2.post1\nmatplotlib==3.2.1\njoblib==0.14.1\nsix==1.14.0\npython-dateutil==2.8.1\npytz==2019.3\ncycler==0.10.0\nkiwisolver==1.1.0\npyparsing==2.4.6\n', 'research/reqcheck_ml.py': '"""ReqCheck research implementation. Python 3.7 / scikit-learn 0.22 compatible."""\nimport re\nimport numpy as np\nfrom scipy.sparse import csr_matrix\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.pipeline import Pipeline, FeatureUnion\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.naive_bayes import MultinomialNB\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.svm import LinearSVC\n\nVAGUE = (\'quickly\', \'easy\', \'user friendly\', \'adequate\', \'efficient\', \'as soon as possible\', \'etc\')\nclass SurfaceFeatures(BaseEstimator, TransformerMixin):\n    """Fixed nonnegative features; no corpus statistics fitted outside folds."""\n    def __init__(self, group=\'all\'):\n        self.group = group\n    def fit(self, X, y=None):\n        return self\n    def transform(self, X):\n        rows=[]\n        for text in X:\n            s=str(text).lower(); words=re.findall(r\'\\b\\w+\\b\',s); n=max(len(words),1)\n            structure=[min(n,100)/100., min(len(s),1000)/1000., min(s.count(\',\')+s.count(\';\'),10)/10.]\n            modality=[sum(w in (\'shall\',\'must\',\'should\',\'may\',\'will\') for w in words)/n,\n                      sum(w in (\'not\',\'never\',\'no\') for w in words)/n,\n                      sum(w.endswith(\'ing\') for w in words)/n]\n            constraints=[sum(bool(re.search(r\'\\d\',w)) for w in words)/n,\n                         float(bool(re.search(r\'\\b(seconds?|minutes?|hours?|milliseconds?|percent|users?)\\b|%\',s))),\n                         sum(term in s for term in VAGUE)/float(len(VAGUE))]\n            values={\'structure\':structure,\'modality\':modality,\'constraints\':constraints}\n            row=[]\n            for name in (\'structure\',\'modality\',\'constraints\'):\n                if self.group == \'all\' or self.group != \'without_\'+name: row.extend(values[name])\n            rows.append(row)\n        return csr_matrix(np.asarray(rows,dtype=float))\n\ndef make_pipeline(name):\n    word=TfidfVectorizer(ngram_range=(1,2),min_df=1,max_features=15000,sublinear_tf=True)\n    char=TfidfVectorizer(analyzer=\'char_wb\',ngram_range=(3,5),min_df=1,max_features=20000,sublinear_tf=True)\n    if name.startswith(\'wordchar\'):\n        feats=FeatureUnion([(\'word\',word),(\'char\',char)])\n    elif name.startswith(\'hybrid\') or name.startswith(\'ablate\'):\n        group=\'all\' if name.startswith(\'hybrid\') else name.split(\':\')[1]\n        feats=FeatureUnion([(\'word\',word),(\'surface\',SurfaceFeatures(group))])\n    else: feats=word\n    if name==\'word_nb\': clf=MultinomialNB(); grid={\'clf__alpha\':[0.5,1.]}\n    elif name.endswith(\'_lr\'): clf=LogisticRegression(solver=\'liblinear\',class_weight=\'balanced\',max_iter=2000,random_state=42); grid={\'clf__C\':[0.1,1.,10.]}\n    else: clf=LinearSVC(class_weight=\'balanced\',random_state=42,max_iter=10000); grid={\'clf__C\':[0.1,1.,10.]}\n    return Pipeline([(\'features\',feats),(\'clf\',clf)]),grid\n\ndef normalize(text):\n    return re.sub(r\'\\W+\',\' \',str(text).lower()).strip()\n\ndef ambiguity_flags(text):\n    s=text.lower(); flags=[]\n    for term in VAGUE:\n        for m in re.finditer(r\'\\b\'+re.escape(term)+r\'\\b\',s):\n            flags.append({\'rule\':\'VAGUE_TERM\',\'term\':term,\'start\':m.start(),\'end\':m.end(),\n                          \'suggestion\':\'Ask the author to specify a measurable acceptance criterion.\'})\n    return flags\n', 'research/run_experiments.py': '"""Nested group CV. Input CSV: requirement_id,project_id,text,label. No GPU."""\nimport os\nos.environ.setdefault(\'OMP_NUM_THREADS\',\'1\')\nos.environ.setdefault(\'OPENBLAS_NUM_THREADS\',\'1\')\nimport argparse,hashlib,json,platform,time,subprocess\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport sklearn, scipy, joblib\nfrom sklearn.model_selection import GroupKFold,GridSearchCV\nfrom sklearn.metrics import f1_score,precision_score,recall_score,accuracy_score,confusion_matrix,classification_report,make_scorer\nfrom reqcheck_ml import make_pipeline,normalize\n\nALLOWED={\'F\',\'A\',\'L\',\'LF\',\'MN\',\'O\',\'PE\',\'SC\',\'SE\',\'US\',\'FT\',\'PO\'}\ndef load_data(path,task):\n    d=pd.read_csv(path,dtype=str).fillna(\'\')\n    required={\'requirement_id\',\'project_id\',\'text\',\'label\'}\n    if not required.issubset(d.columns): raise ValueError(\'Required columns: \'+\',\'.join(sorted(required)))\n    if d.empty or any(d[c].str.strip().eq(\'\').any() for c in required): raise ValueError(\'Empty dataset or required values\')\n    if d.requirement_id.duplicated().any(): raise ValueError(\'requirement_id must be unique\')\n    d[\'label\']=d.label.str.strip().str.upper()\n    if set(d.label)-ALLOWED: raise ValueError(\'Unknown labels: \'+str(set(d.label)-ALLOWED))\n    d[\'normalized\']=d.text.map(normalize)\n    if d.normalized.eq(\'\').any(): raise ValueError(\'Blank normalized requirement\')\n    if d.groupby(\'normalized\').label.nunique().max()>1: raise ValueError(\'Conflicting duplicate labels: adjudicate before running\')\n    before=len(d); d=d.drop_duplicates(\'normalized\').copy(); removed_duplicates=before-len(d)\n    if task==\'binary\':d[\'target\']=np.where(d.label==\'F\',\'F\',\'NFR\')\n    else:d=d[d.label!=\'F\'].copy(); d[\'target\']=d.label\n    d=d.sort_values([\'project_id\',\'requirement_id\']).reset_index(drop=True)\n    if d.target.nunique()<2:raise ValueError(\'Task needs at least two classes\')\n    return d,removed_duplicates,before\n\ndef run(args):\n    out=Path(args.output);out.mkdir(parents=True,exist_ok=True)\n    d,removed_duplicates,original=load_data(args.data,args.task)\n    # Keep an exact record of retained rows; no automatic rare-class merging.\n    d.to_csv(str(out/\'cleaned_data.csv\'),index=False)\n    labels=sorted(d.target.unique()); X=d.text.values;y=d.target.values;groups=d.project_id.values\n    if len(set(groups))<3:raise ValueError(\'At least 3 distinct projects required for nested group CV\')\n    outer=list(GroupKFold(n_splits=min(5,len(set(groups)))).split(X,y,groups))\n    for tr,te in outer:\n        if len(set(y[tr]))<2:raise ValueError(\'Outer training fold has fewer than 2 classes\')\n    scorer=make_scorer(f1_score,average=\'macro\',labels=labels,zero_division=0)\n    names=[\'word_nb\',\'word_lr\',\'word_svm\',\'wordchar_lr\',\'wordchar_svm\',\'hybrid_lr\',\'hybrid_svm\',\n           \'ablate:without_structure\',\'ablate:without_modality\',\'ablate:without_constraints\']\n    if args.quick:names=[\'word_svm\',\'hybrid_svm\']\n    results=[];predictions=[];splits=[]\n    for fold,(tr,te) in enumerate(outer):\n        assert not(set(groups[tr])&set(groups[te]))\n        splits.append({\'fold\':fold,\'train_ids\':d.iloc[tr].requirement_id.tolist(),\'test_ids\':d.iloc[te].requirement_id.tolist()})\n        inner=list(GroupKFold(n_splits=min(3,len(set(groups[tr])))).split(X[tr],y[tr],groups[tr]))\n        for itr,ite in inner:\n            if len(set(y[tr][itr]))<2:raise ValueError(\'Inner training fold has fewer than 2 classes; revise documented protocol\')\n        for name in names:\n            pipe,grid=make_pipeline(name); t=time.perf_counter()\n            search=GridSearchCV(pipe,grid,cv=inner,scoring=scorer,n_jobs=1,error_score=\'raise\',refit=True)\n            search.fit(X[tr],y[tr]);fit_seconds=time.perf_counter()-t\n            t=time.perf_counter();pred=search.predict(X[te]);prediction_seconds=time.perf_counter()-t\n            row={\'model\':name,\'fold\':fold,\'macro_f1\':f1_score(y[te],pred,labels=labels,average=\'macro\',zero_division=0),\n                 \'macro_precision\':precision_score(y[te],pred,labels=labels,average=\'macro\',zero_division=0),\n                 \'macro_recall\':recall_score(y[te],pred,labels=labels,average=\'macro\',zero_division=0),\n                 \'accuracy\':accuracy_score(y[te],pred),\'fit_seconds\':fit_seconds,\'predict_seconds\':prediction_seconds,\n                 \'test_n\':len(te),\'missing_test_classes\':\',\'.join(sorted(set(labels)-set(y[te]))),\n                 \'missing_train_classes\':\',\'.join(sorted(set(labels)-set(y[tr]))),\'best_params\':json.dumps(search.best_params_)}\n            results.append(row)\n            for idx,p in zip(te,pred):predictions.append({\'model\':name,\'fold\':fold,\'requirement_id\':d.iloc[idx].requirement_id,\n                          \'project_id\':groups[idx],\'true\':y[idx],\'predicted\':p})\n            print(name,\'fold\',fold,\'macro-F1\',round(row[\'macro_f1\'],4),flush=True)\n    res=pd.DataFrame(results);res.to_csv(str(out/\'fold_metrics.csv\'),index=False)\n    pp=pd.DataFrame(predictions);pp.to_csv(str(out/\'oof_predictions.csv\'),index=False)\n    summary=res.groupby(\'model\').macro_f1.agg([\'mean\',\'std\']).sort_values(\'mean\',ascending=False)\n    summary.to_csv(str(out/\'summary.csv\'))\n    # OOF metrics and confusion matrices: each retained requirement predicted once per approach.\n    pooled=[]\n    for name in names:\n        z=pp[pp.model==name]\n        pooled.append({\'model\':name,\'pooled_macro_f1\':f1_score(z.true,z.predicted,labels=labels,average=\'macro\',zero_division=0)})\n        report=classification_report(z.true,z.predicted,labels=labels,output_dict=True,zero_division=0)\n        (out/(name.replace(\':\',\'_\')+\'_report.json\')).write_text(json.dumps(report,indent=2))\n        cm=confusion_matrix(z.true,z.predicted,labels=labels)\n        fig,ax=plt.subplots(figsize=(max(5,len(labels)*.65),max(4,len(labels)*.55)))\n        im=ax.imshow(cm,cmap=\'Blues\');fig.colorbar(im,ax=ax)\n        ax.set(xticks=range(len(labels)),yticks=range(len(labels)),xticklabels=labels,yticklabels=labels,xlabel=\'Predicted\',ylabel=\'Actual\',title=name)\n        for i in range(len(labels)):\n            for j in range(len(labels)):ax.text(j,i,str(cm[i,j]),ha=\'center\',va=\'center\',fontsize=8)\n        fig.tight_layout();fig.savefig(str(out/(name.replace(\':\',\'_\')+\'_confusion.png\')),dpi=160);plt.close(fig)\n    pd.DataFrame(pooled).to_csv(str(out/\'pooled_metrics.csv\'),index=False)\n    fig,ax=plt.subplots(figsize=(10,5));ax.barh(summary.index,summary[\'mean\'],xerr=summary[\'std\'].fillna(0));ax.set(xlim=(0,1),xlabel=\'Outer-fold macro-F1 (mean +/- SD)\',title=\'Grouped nested cross-validation\');fig.tight_layout();fig.savefig(str(out/\'comparison.png\'),dpi=180);plt.close(fig)\n    if {\'word_svm\',\'hybrid_svm\'}.issubset(names):\n        diff=res.pivot(index=\'fold\',columns=\'model\',values=\'macro_f1\');diff[\'hybrid_minus_word\']=diff.hybrid_svm-diff.word_svm\n        diff[[\'word_svm\',\'hybrid_svm\',\'hybrid_minus_word\']].to_csv(str(out/\'paired_fold_differences.csv\'))\n    # Deployment selection is exploratory; do not call its maximum CV score an independent final test.\n    selected=summary.index[0];pipe,grid=make_pipeline(selected)\n    final_cv=list(GroupKFold(n_splits=min(3,len(set(groups)))).split(X,y,groups))\n    search=GridSearchCV(pipe,grid,cv=final_cv,scoring=scorer,n_jobs=1,error_score=\'raise\');search.fit(X,y)\n    joblib.dump(search.best_estimator_,str(out/\'model.joblib\'))\n    manifest={\'status\':\'DEMONSTRATION_ONLY\' if args.demo else \'EXPERIMENT_RUN_REQUIRES_REVIEW\',\n              \'task\':args.task,\'original_rows\':original,\'removed_exact_duplicates\':removed_duplicates,\'task_rows_after_dedup\':len(d),\'labels\':labels,\n              \'class_counts\':d.target.value_counts().to_dict(),\'projects\':len(set(groups)),\'selected_for_deployment\':selected,\n              \'selected_params\':search.best_params_,\'seed\':42,\'outer_splits\':len(outer),\'inner_splits_max\':3,\n              \'dataset_sha256\':hashlib.sha256(Path(args.data).read_bytes()).hexdigest(),\n              \'python\':platform.python_version(),\'sklearn\':sklearn.__version__,\'numpy\':np.__version__,\n              \'scipy\':scipy.__version__,\'pandas\':pd.__version__,\'matplotlib\':matplotlib.__version__,\n              \'source_sha256\':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in [Path(__file__),Path(__file__).with_name(\'reqcheck_ml.py\')]},\n              \'interpretation\':\'Fixed global label macro-F1. Absent fold classes count as zero. Inspect support tables. Selected-model score is selection-biased; evaluate externally for a final claim.\'}\n    (out/\'manifest.json\').write_text(json.dumps(manifest,indent=2));(out/\'splits.json\').write_text(json.dumps(splits,indent=2))\n    (out/\'environment_freeze.txt\').write_text(subprocess.check_output([os.sys.executable,\'-m\',\'pip\',\'freeze\']).decode())\n    print(summary.to_string());print(\'Saved\',out)\nif __name__==\'__main__\':\n    ap=argparse.ArgumentParser();ap.add_argument(\'--data\',required=True);ap.add_argument(\'--output\',default=\'results\');ap.add_argument(\'--task\',choices=[\'binary\',\'nfr\'],default=\'binary\');ap.add_argument(\'--quick\',action=\'store_true\');ap.add_argument(\'--demo\',action=\'store_true\');run(ap.parse_args())\n', 'research/convert_promise.py': '"""Convert original PROMISE three-column ARFF without scipy\'s string-ARFF limitation."""\nimport re,csv,argparse\nfrom pathlib import Path\np=argparse.ArgumentParser();p.add_argument(\'source\');p.add_argument(\'destination\');p.add_argument(\'--encoding\',default=\'utf-8-sig\');a=p.parse_args()\nrows=[];data=False\nfor number,line in enumerate(Path(a.source).read_text(encoding=a.encoding).splitlines(),1):\n    s=line.strip()\n    if s.lower()==\'@data\':data=True;continue\n    if not data or not s or s.startswith(\'%\'):continue\n    m=re.match(r\'\'\'^(\\d+)\\s*,\\s*([\'"])(.*)\\2\\s*,\\s*([A-Za-z]+)\\s*(?:%.*)?$\'\'\',s)\n    if not m:raise ValueError(\'Unsupported ARFF row %d; inspect manually rather than skipping it\'%number)\n    rows.append({\'requirement_id\':\'PROMISE-%04d\'%(len(rows)+1),\'project_id\':m.group(1),\'text\':m.group(3).replace("\\\\\'", "\'"),\'label\':m.group(4).upper()})\nif not rows:raise ValueError(\'No data rows\')\nwith open(a.destination,\'w\',newline=\'\',encoding=\'utf-8\') as f:\n    w=csv.DictWriter(f,fieldnames=[\'requirement_id\',\'project_id\',\'text\',\'label\']);w.writeheader();w.writerows(rows)\nprint(\'Converted\',len(rows),\'rows. Verify counts and acknowledge the original dataset authors.\')\n', 'research/evaluate_external.py': '"""Evaluate an untouched external CSV once after freezing the model and protocol."""\nimport argparse,json,hashlib\nfrom pathlib import Path\nimport joblib\nfrom sklearn.metrics import classification_report,confusion_matrix\nfrom run_experiments import load_data\np=argparse.ArgumentParser();p.add_argument(\'--model\',required=True);p.add_argument(\'--data\',required=True);p.add_argument(\'--task\',choices=[\'binary\',\'nfr\'],default=\'binary\');p.add_argument(\'--output\',required=True);a=p.parse_args()\nm=joblib.load(a.model);d,_,_=load_data(a.data,a.task);pred=m.predict(d.text.values)\n# Explicitly include unseen external labels; report class coverage rather than silently dropping them.\nlabels=sorted(set(d.target)|set(m.classes_));out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\nreport=classification_report(d.target,pred,labels=labels,output_dict=True,zero_division=0)\nreport[\'dataset_sha256\']=hashlib.sha256(Path(a.data).read_bytes()).hexdigest();report[\'model_sha256\']=hashlib.sha256(Path(a.model).read_bytes()).hexdigest()\n(out/\'external_report.json\').write_text(json.dumps(report,indent=2));d[\'predicted\']=pred;d.to_csv(str(out/\'external_predictions.csv\'),index=False)\nprint(json.dumps(report,indent=2))\n', 'data/demo_requirements.csv': 'requirement_id,project_id,text,label\nDEMO-1-0,1,The user shall create a booking for service 1.,F\nDEMO-1-1,1,The administrator can export invoice 1.,F\nDEMO-1-2,1,The system shall allow editing record number 1.,F\nDEMO-1-3,1,The system shall respond within 100 milliseconds.,PE\nDEMO-1-4,1,The server must process 20 requests per second.,PE\nDEMO-1-5,1,The service shall support 100 concurrent users.,PE\nDEMO-1-6,1,Passwords for role 1 must be encrypted.,SE\nDEMO-1-7,1,Account 1 shall require two factor authentication.,SE\nDEMO-1-8,1,Access to resource 1 shall be restricted to authorized users.,SE\nDEMO-2-0,2,The user shall create a booking for service 2.,F\nDEMO-2-1,2,The administrator can export invoice 2.,F\nDEMO-2-2,2,The system shall allow editing record number 2.,F\nDEMO-2-3,2,The system shall respond within 200 milliseconds.,PE\nDEMO-2-4,2,The server must process 40 requests per second.,PE\nDEMO-2-5,2,The service shall support 200 concurrent users.,PE\nDEMO-2-6,2,Passwords for role 2 must be encrypted.,SE\nDEMO-2-7,2,Account 2 shall require two factor authentication.,SE\nDEMO-2-8,2,Access to resource 2 shall be restricted to authorized users.,SE\nDEMO-3-0,3,The user shall create a booking for service 3.,F\nDEMO-3-1,3,The administrator can export invoice 3.,F\nDEMO-3-2,3,The system shall allow editing record number 3.,F\nDEMO-3-3,3,The system shall respond within 300 milliseconds.,PE\nDEMO-3-4,3,The server must process 60 requests per second.,PE\nDEMO-3-5,3,The service shall support 300 concurrent users.,PE\nDEMO-3-6,3,Passwords for role 3 must be encrypted.,SE\nDEMO-3-7,3,Account 3 shall require two factor authentication.,SE\nDEMO-3-8,3,Access to resource 3 shall be restricted to authorized users.,SE\nDEMO-4-0,4,The user shall create a booking for service 4.,F\nDEMO-4-1,4,The administrator can export invoice 4.,F\nDEMO-4-2,4,The system shall allow editing record number 4.,F\nDEMO-4-3,4,The system shall respond within 400 milliseconds.,PE\nDEMO-4-4,4,The server must process 80 requests per second.,PE\nDEMO-4-5,4,The service shall support 400 concurrent users.,PE\nDEMO-4-6,4,Passwords for role 4 must be encrypted.,SE\nDEMO-4-7,4,Account 4 shall require two factor authentication.,SE\nDEMO-4-8,4,Access to resource 4 shall be restricted to authorized users.,SE\nDEMO-5-0,5,The user shall create a booking for service 5.,F\nDEMO-5-1,5,The administrator can export invoice 5.,F\nDEMO-5-2,5,The system shall allow editing record number 5.,F\nDEMO-5-3,5,The system shall respond within 500 milliseconds.,PE\nDEMO-5-4,5,The server must process 100 requests per second.,PE\nDEMO-5-5,5,The service shall support 500 concurrent users.,PE\nDEMO-5-6,5,Passwords for role 5 must be encrypted.,SE\nDEMO-5-7,5,Account 5 shall require two factor authentication.,SE\nDEMO-5-8,5,Access to resource 5 shall be restricted to authorized users.,SE\nDEMO-6-0,6,The user shall create a booking for service 6.,F\nDEMO-6-1,6,The administrator can export invoice 6.,F\nDEMO-6-2,6,The system shall allow editing record number 6.,F\nDEMO-6-3,6,The system shall respond within 600 milliseconds.,PE\nDEMO-6-4,6,The server must process 120 requests per second.,PE\nDEMO-6-5,6,The service shall support 600 concurrent users.,PE\nDEMO-6-6,6,Passwords for role 6 must be encrypted.,SE\nDEMO-6-7,6,Account 6 shall require two factor authentication.,SE\nDEMO-6-8,6,Access to resource 6 shall be restricted to authorized users.,SE\nDEMO-7-0,7,The user shall create a booking for service 7.,F\nDEMO-7-1,7,The administrator can export invoice 7.,F\nDEMO-7-2,7,The system shall allow editing record number 7.,F\nDEMO-7-3,7,The system shall respond within 700 milliseconds.,PE\nDEMO-7-4,7,The server must process 140 requests per second.,PE\nDEMO-7-5,7,The service shall support 700 concurrent users.,PE\nDEMO-7-6,7,Passwords for role 7 must be encrypted.,SE\nDEMO-7-7,7,Account 7 shall require two factor authentication.,SE\nDEMO-7-8,7,Access to resource 7 shall be restricted to authorized users.,SE\nDEMO-8-0,8,The user shall create a booking for service 8.,F\nDEMO-8-1,8,The administrator can export invoice 8.,F\nDEMO-8-2,8,The system shall allow editing record number 8.,F\nDEMO-8-3,8,The system shall respond within 800 milliseconds.,PE\nDEMO-8-4,8,The server must process 160 requests per second.,PE\nDEMO-8-5,8,The service shall support 800 concurrent users.,PE\nDEMO-8-6,8,Passwords for role 8 must be encrypted.,SE\nDEMO-8-7,8,Account 8 shall require two factor authentication.,SE\nDEMO-8-8,8,Access to resource 8 shall be restricted to authorized users.,SE\n', 'data/promise_reviewed.csv': 'requirement_id,project_id,text,label\nPROMISE-0001,1,The system shall refresh the display every 60 seconds.,PE\nPROMISE-0002,1,The application shall match the color of the schema set forth by Department of Homeland Security,LF\nPROMISE-0003,1, If projected  the data must be readable.  On a 10x10 projection screen  90% of viewers must be able to read Event / Activity data from a viewing distance of 30,US\nPROMISE-0004,1, The product shall be available during normal business hours. As long as the user has access to the client PC  the system will be available 99% of the time during the first six months of operation.,A\nPROMISE-0005,1, If projected  the data must be understandable. On a 10x10 projection screen  90% of viewers must be able to determine that Events or Activities are occuring in current time from a viewing distance of 100,US\nPROMISE-0006,1,The product shall ensure that it can only be accessed by authorized users.  The product will be able to distinguish between authorized and unauthorized users in all access attempts,SE\nPROMISE-0007,1,The product shall be intuitive and self-explanatory.  :  90% of new users shall be able to start the display of Events or Activities within 90 minutes of using the product.,US\nPROMISE-0008,1,The product shall respond fast to keep up-to-date data in the display.,PE\nPROMISE-0009,1,The system shall have a MDI form that allows for the viewing of the graph and the data table.,F\nPROMISE-0010,1,The system shall display Events in a vertical table by time.,F\nPROMISE-0011,1,The system shall display the Events in a graph by time.,F\nPROMISE-0012,1,The system shall display Events or Activities.,F\nPROMISE-0013,1, The display shall have two regions:  left 2/3 of the display is graphical  right 1/3 of the display is a data table,F\nPROMISE-0014,1,The data displayed in both the nodes within the graph and the rows in the table are MSEL Summary data,F\nPROMISE-0015,1,The table side of the display shall be split into 2 regions: sequential and temporal.,F\nPROMISE-0016,1,The top 1/4 of the table will hold events that are to occur sequentially.,F\nPROMISE-0017,1,The bottom 3/4 of the table will hold events that occur according to its relevance to current time.,F\nPROMISE-0018,1,The system shall color code events according to their variance from current time.,F\nPROMISE-0019,1,The system shall display data from the Sync Matrix 1.0 and Exercise Management Tool 1.0 applications,F\nPROMISE-0020,1,The system shall link Events back to either the Sync Matrix 1.0 or the Exercise Managment Tool 1.0 applications for modifications.,F\nPROMISE-0021,1,The system shall allow modification of the display.,F\nPROMISE-0022,1,The system shall offer a display of all the Events in the exercise.,F\nPROMISE-0023,1,The system shall filter data by: Venues and Key Events.,F\nPROMISE-0024,1,The system shall allow a user to define the time segments,F\nPROMISE-0025,1,The system shall display the local and exercise time in separate clocks,F\nPROMISE-0026,1,The system shall offer the ability to pause and resume the refresh of data.,F\nPROMISE-0027,1,The system shall provide charts for the Activity or Event actual versus assumed time.,F\nPROMISE-0028,1,The system shall provide a history report of changes made to the Activity or Event data,F\nPROMISE-0029,2,The look and feel of the system shall conform to the user interface standards of the smart device.,LF\nPROMISE-0030,2, The user interface shall have standard menus  buttons for navigation,LF\nPROMISE-0031,2,The system shall have a professional appearance,LF\nPROMISE-0032,2,The product shall have a consistent color scheme and fonts.,LF\nPROMISE-0033,2,The system shall be used by realtors with no training.,US\nPROMISE-0034,2,The system shall be intuitive and self explanatory.,US\nPROMISE-0035,2,The product shall be easy for a realtor to learn.,US\nPROMISE-0036,2,The product shall use symbols and words that are naturally understandable by the realtor community.,US\nPROMISE-0037,2,The product shall produce search results in an acceptable time,PE\nPROMISE-0038,2,The search results shall be returned no later 30      seconds  after the user has entered the search criteria,PE\nPROMISE-0039,2,The product shall generate a CMA report in an acceptable time.,PE\nPROMISE-0040,2,The CMA report shall be returned no later 60 seconds after the user has entered the CMA report criteria.,PE\nPROMISE-0041,2,The product shall synchronize contacts and appointments in an acceptable time.,PE\nPROMISE-0042,2,The product shall synchronize with the office system every hour.,PE\nPROMISE-0043,2, The product shall be available for use 24 hours per day  365 days  per year,A\nPROMISE-0044,2, Out of 1000 accesses to the system  the system is available 999 times.,A\nPROMISE-0045,2,The product shall operate in offline mode whenever internet connection is unavailable.,FT\nPROMISE-0046,2, The product shall allow the user to view previously downloaded search results  CMA reports  and appointments.,FT\nPROMISE-0047,2,The product shall retain user preferences in the event of a failure.,FT\nPROMISE-0048,2,100% of saved user preferences shall be restored when system comes back online.,FT\nPROMISE-0049,2,The product shall be able to support multiple remote users,SC\nPROMISE-0050,2,The product shall be able support 1000 simultaneous users.,SC\nPROMISE-0051,2,The product shall be capable of handling the existing 1000 users. This number is expected to grow 5 times within the next year.,SC\nPROMISE-0052,2,The product shall be installed by an untrained realtor   without recourse to separately-printed instructions.,US\nPROMISE-0053,2,90% of untrained realtors shall be able to install the product on their device without printed instructions,US\nPROMISE-0054,2,The product is expected to run on Windows CE and Palm operating systems.,PO\nPROMISE-0055,2,Only registered realtors shall be able to access the system.,SE\nPROMISE-0056,2,Every user of the system shall be authenticated and authorized.,SE\nPROMISE-0057,2,The product shall prevent its data from incorrect data being introduced.,SE\nPROMISE-0058,2,The system shall allow a real estate agent to query MLS information,F\nPROMISE-0059,2,The user shall be able to get driving directions to a selected property,F\nPROMISE-0060,2,The user shall be able to download appointments and contact information for clients,F\nPROMISE-0061,2,The system shall display clear property images in the search results,F\nPROMISE-0062,2,The system shall update or create new property listings in the MLS,F\nPROMISE-0063,2,Realtor shall be able to generate a CMA report based on property criteria,F\nPROMISE-0064,2,The system shall be able to call the seller or buyer to schedule an appointment,F\nPROMISE-0065,2,The realtor shall be notified of new client appointments after automatic synchronization with office system,F\nPROMISE-0066,2,The realtor shall be able to retrieve a map showing property locations for a specified area,F\nPROMISE-0067,2,The system shall notify the realtor when a seller or buyer responds to an appointment request,F\nPROMISE-0068,2,The product shall allow the user to save the property search results.,F\nPROMISE-0069,3,The system shall be easy to use by the Program Administrators/Nursing Staff Members.,US\nPROMISE-0070,3,All of the users shall be able to use the system after one day of training.,US\nPROMISE-0071,3,The system shall help the user avoid making mistakes while scheduling classes and clinicals for the nursing students.,US\nPROMISE-0072,3,The number of mistakes noted by the students shall be decreased by 50% in the first year.,US\nPROMISE-0073,3,The system shall be intuitive to the Program Administrators/Nursing Staff Members.,US\nPROMISE-0074,3,2 out of 3 Program Administrators/Nursing Staff Members shall successfully be able to use the system to manage the scheduling of classes and clinicals.,US\nPROMISE-0075,3,The system shall use symbols and words that are naturally understandable by the user community.,US\nPROMISE-0076,3,The verbiage used within the system shall be consistent with Standard English and the terminology of the national Nursing Community.,US\nPROMISE-0077,3,Staff members shall be able to complete a set of tasks in a timely manner.,PE\nPROMISE-0078,3,The response time of general student management tasks shall take no longer than 5 seconds and the response time of schedule generation shall take no longer than 30 seconds.,PE\nPROMISE-0079,3,The system shall be available for use between the hours of 8am and 6pm.,A\nPROMISE-0080,3,The system shall achieve 95% up time.,A\nPROMISE-0081,3,system shall be able to handle all of the user requests/usage during business hours.,SC\nPROMISE-0082,3,The system shall cater to 10 simultaneous users from 9am to 5pm.,SC\nPROMISE-0083,3,The system shall be capable of processing 100% of nursing students and their classes for the next 10 years.,SC\nPROMISE-0084,3,The system shall be expected to manage the nursing program curriculum and class/clinical scheduling for a minimum of 5 years.,SC\nPROMISE-0085,3,The system shall able to operate within a business office environment typical of the Nursing Department at DePaul University.,O\nPROMISE-0086,3,The system shall be used within the specifications defined by the computers used by the Program Administrators/Nursing Staff Members.,O\nPROMISE-0087,3,The system shall operate within the Windows XP Professional operating system.,O\nPROMISE-0088,3,The system shall interface with CampusConnect’s central server,O\nPROMISE-0089,3,The system shall interface with the faculty central server.,O\nPROMISE-0090,3,The system shall interface with the student master server.,O\nPROMISE-0091,3,We must be able to interface with any HTML browser.,O\nPROMISE-0092,3,Only authorized users shall have access to clinical site information.,SE\nPROMISE-0093,3,Program Administrators/Nursing Staff Members are the only people who shall have access to clinical site details.,SE\nPROMISE-0094,3,Only authorized users shall have access to students’ personal information.,SE\nPROMISE-0095,3,Dr Susan Poslusny and Dr Julie Donalek are the only people who shall have access to students’ personal information.,SE\nPROMISE-0096,3,Only authorized users shall have access to the portion of the system that interfaces with CampusConnect.,SE\nPROMISE-0097,3,Fit Criterion: Dr Susan Poslusny and Karen Sysol are the only people who shall have access to the final class section scheduling for the system that interfaces with CampusConnect.,SE\nPROMISE-0098,3,The system shall have basic data integrity checking to reduce the possibility of incorrect or invalid data being introduced.,SE\nPROMISE-0099,3,The system shall protect private information in accordance with the organization’s information policy.,SE\nPROMISE-0100,3,The system shall be built such that it is as secure as possible from malicious interference.,SE\nPROMISE-0101,3,The system shall use the organization’s standard virus protection software to help monitor threats to itself.,SE\nPROMISE-0102,3,A cohort shall be associated to one Program of Study offered by the department.,F\nPROMISE-0103,3,Program Administrators and Nursing Staff Members shall be able to add a listing of students to a cohort.,F\nPROMISE-0104,3,Program Administrators and Nursing Staff Members shall be able to add a single student to a cohort,F\nPROMISE-0105,3,Students added to a cohort shall be eligible for the cohort’s Program of Study.,F\nPROMISE-0106,3,Program Administrators and Nursing Staff Members shall be able to specify the preferred sequence of classes for a given cohort.,F\nPROMISE-0107,3, For each class within a sequence for a cohort  Program Administrators and Nursing Staff Members shall be able to specify which quarter that class will be offered.,F\nPROMISE-0108,3, For each class within a sequence for a cohort  Program Administrators and Nursing Staff Members shall be able to specify which quarter that class will be offered.,F\nPROMISE-0109,3,Program Administrators and Nursing Staff Members shall be able to add new classes to a sequence of classes,F\nPROMISE-0110,3,Program Administrators and Nursing Staff Members shall be able to add clinical classes or sections to a sequence of classes,F\nPROMISE-0111,3, Program Administrators and Nursing Staff Members shall be able to add a new cohort to the system  identified by start month and year.,F\nPROMISE-0112,3,Program Administrators and Nursing Staff Members shall be able to add new classes for the nursing department into the system.,F\nPROMISE-0113,3,A class shall be either a non-clinical class or a clinical class.,F\nPROMISE-0114,3, A non-clinical class shall specify the course name  lecture room requirements  and instructor needs.,F\nPROMISE-0115,3, A clinical class shall specify the course name  lecture room requirements  clinical site needs  lecture instructor needs  and clinical lab instructor needs.,F\nPROMISE-0116,3,Program Administrators/Nursing Staff Members shall be able to create a new Program of Study.,F\nPROMISE-0117,3,A Program of Study shall consist of a program name and listing of required classes (both clinical and non-clinical) that must be completed.,F\nPROMISE-0118,3,Program Administrators and Nursing Staff Members shall have the ability to specify which classes are required for a Program of Study,F\nPROMISE-0119,3,The system shall be able to display a report of needed classes for a given quarter for all cohorts of all programs for Program Administrators/Nursing Staff Members’ planning purposes.,F\nPROMISE-0120,3, The report of needed classes shall include (but not be limited to) classes to be offered  number of sections needed  number of labs needed  and room types needed.,F\nPROMISE-0121,3,Classes for a given cohort shall not conflict with regards to the time and day that they are offered.,F\nPROMISE-0122,3,Program Administrators and Nursing Staff Members shall be able to add a new clinical site into the system.,F\nPROMISE-0123,3, Clinical site information shall include (but not be limited to) the name of the site  contact person  and contact information.,F\nPROMISE-0124,3,Program Administrators and Nursing Staff Members shall be able to add a new clinical lab section for an existing clinical class into the System.,F\nPROMISE-0125,3, A clinical lab section shall include the clinical site name  the class  instructor  day and time of the lab.,F\nPROMISE-0126,3,Program Administrators and Nursing Staff Members shall be able to add a student who has registered for a clinical class to a clinical lab section for that class.,F\nPROMISE-0127,3,The system shall allow a Program Administrator or Nursing staff member to remove a student from a clinical lab section.,F\nPROMISE-0128,3,The system shall allow a Program Administrator/Nursing Staff Member to move a student from one clinical lab section to another clinical lab section corresponding to the same clinical class.,F\nPROMISE-0129,3,Program Administrators/Nursing Staff Members shall be able to cancel a clinical lab section only if there are no students registered for that clinical lab section.,F\nPROMISE-0130,3,A Program Administrator/Nursing Staff Member shall be able to add a new nursing student to the system.,F\nPROMISE-0131,3, A staff member shall have the ability to manage the progress of a part time student  which includes modifying their sequence of classes and timeline of classes.,F\nPROMISE-0132,3, The system shall be able to display a printable summary for individual cohorts  which will include the students enlisted  the Program of study  sequence of classes  cohort progress through the program  and timeline of completion.,F\nPROMISE-0133,3, The system shall be able to display a printable summary for individual clinical sites  which will include information on labs being held at that site for the given quarter (Clinical site  general information on labs held at that location  dates and times of all labs held there).,F\nPROMISE-0134,3, The system shall be able to display a printable summary for individual clinical labs  which will include (but not be limited to) class name  class number  lab section  listing of students enrolled  instructor  day(s)  time  and location.,F\nPROMISE-0135,3, The system shall be able to display a printable summary for individual nursing students  which will include (but not be limited to) student name  student ID  admission date  classes  credits  GPA  and the cohort that the student is enrolled in.,F\nPROMISE-0136,3,The system shall be able to display a printable summary of the system log.,F\nPROMISE-0137,3, The system shall contain contact information (e-mail and phone number) for all people relevant to the system  including (but not limited to) staff members  students  lecture instructors  clinical lab instructors  and clinical site administrators.,F\nPROMISE-0138,3,The system will use the stored e-mail addresses as a primary means of communicating information to affected parties.,F\nPROMISE-0139,3, The system will notify affected parties for routine communication purposes  including but not limited to informing students of upcoming offered classes (timeline) and what their schedule will be/shall be.,F\nPROMISE-0140,3, The system will notify affected parties when changes occur affecting clinicals  including but not limited to clinical section capacity changes  and clinical section cancellations.,F\nPROMISE-0141,3, The system will notify affected parties when changes occur affecting classes  including but not limited to class cancellations  class section detail changes  and changes to class offerings for a given quarter.,F\nPROMISE-0142,3, The system will notify affected parties when changes occur affecting cohorts  including but not limited to changes to the sequence for a cohort’s program of study and changes to a given week’s schedule (lab cancelled this week due to instructor illness).,F\nPROMISE-0143,3, The system will notify affected parties when changes occur affecting specific students  including but not limited to changing a student’s status from full time to part time and moving a student to a different clinical section.,F\nPROMISE-0144,3, Program Administrators/Nursing Staff Members shall have the ability to modify information relating to cohorts  including cohort identifier  program of study  preferred sequence of classes and quarters that a cohort will be taking specific classes.,F\nPROMISE-0145,3, Program Administrators/Nursing Staff Members shall have the ability to modify information relating to a Program of Study within the Nursing Department  including the Program of study name  and required classes for that Program of Study.,F\nPROMISE-0146,3, Program Administrators/Nursing Staff Members shall have the ability to modify information relating to a Clinical Site  including the clinical site name  site contact person  contact information  and address.,F\nPROMISE-0147,3, Program Administrators/Nursing Staff Members shall have the ability to modify information relating to a Clinical Lab Section  including the clinical site name  the department/section for the lab  the class  instructor  day and time of lab  and contact information.,F\nPROMISE-0148,3, Program Administrators/Nursing Staff Members shall have the ability to modify information relating to a Nursing Student  including student ID  student name  phone number  e-mail  status (full time or part time)  and program of study.,F\nPROMISE-0149,4,The Disputes application shall comply with the corporate standards for user interface creation for internally and externally used applications.,LF\nPROMISE-0150,4,All screens created as part of the Disputes application must comply with corporate standards for interface creation.,LF\nPROMISE-0151,4,The list of dispute cases that are displayed after a search is performed must be color coded for easy identification of dispute cases based upon the dispute case status.,US\nPROMISE-0152,4,100% of the cardmember and merchant services representatives shall use the Disputes application regularly after a 2-day training course.,US\nPROMISE-0153,4,100% of cardmember services representatives shall be able to successfully create a dispute case on the first encounter after completing the training course.,US\nPROMISE-0154,4,100% of merchant services representatives shall be able to successfully perform a follow up action on a dispute case on the first encounter after completing the training course.,US\nPROMISE-0155,4,The maximum wait time for a user navigating from one screen to another within the Disputes application shall be no more than 5 seconds.,PE\nPROMISE-0156,4,The Disputes application shall support 350 concurrent users without any degradation of performance in the application.,PE\nPROMISE-0157,4, The Disputes application shall be available 24x7x365  with the exception of the following scheduled maintenance windows.  Monday – Saturday 3:00am to 4:00am EST Sunday 1:00amd to 5:00am EST,O\nPROMISE-0158,4,The Disputes application shall interface with the Statement Database. The Statement Database provides the transaction details to the Disputes System. All transaction details must be obtained from the Statement Database,O\nPROMISE-0159,4,The Disputes application shall interface with the Cardmember Information Database.  The Cardmember Information Database provides detailed information with regard to a cardmember.  All cardmember detail information must be obtained from the Cardmember Information Database.,O\nPROMISE-0160,4,The Disputes applications shall interface with the Merchant Information Database.  The Merchant Information Database provides detailed information with regard to the merchant. All merchant detail information shall be obtained from the Merchant Information Database.,O\nPROMISE-0161,4,The Disputes application shall interface with the Letters application.  This will allow the Disputes application to request letters as part of the dispute initiation and dispute follow up process. All letter requests must be sent to the Print Letter Utility application.,O\nPROMISE-0162,4,The Disputes application shall interface with the Cardmember Posting and Billing System.  This will allow the Disputes application to request adjustments to cardmember and merchant’s accounts. All adjustment requests must be sent to the Cardmember Posting and Billing System.,O\nPROMISE-0163,4,The Disputes application shall ensure that only users with the appropriate access level are allowed to initiate dispute requests.,SE\nPROMISE-0164,4,Only users with Update access must be able to initiate dispute requests.,SE\nPROMISE-0165,4,The Disputes application shall ensure that only authorized users are allowed to logon to the application.,SE\nPROMISE-0166,4, All authorized users  once authenticated must be able to logon to the Disputes System,SE\nPROMISE-0167,4,The Disputes System shall generate a User Report that details any new users that have been added and any changes to an existing user’s access level.  The report will be reviewed for auditing purposes.,SE\nPROMISE-0168,4,All additions of new users and modifications to user’s access shall be recorded on the User Report.,SE\nPROMISE-0169,4, If a user account is revoked  it can only be re-instantiated by the System Administrator.,SE\nPROMISE-0170,4,Only the System Administrator shall be able to reset a revoked user logon account,SE\nPROMISE-0171,4,The Disputes application shall manipulate and view business data. Users will not have direct access to data files or databases.,SE\nPROMISE-0172,4,All updates to data files or database must be initiated from the Disputes System.,SE\nPROMISE-0173,4,The Disputes application must conform to the legal requirements as specified by the Merchant Operating Regulations.,L\nPROMISE-0174,4,All business rules specified in the Disputes System shall be in compliance with the Merchant Operating Regulations.,L\nPROMISE-0175,4,The Disputes application must conform to the legal requirements as specified by Regulation E and Regulation Z that govern credit card disputes processing.,L\nPROMISE-0176,4,All business rules specified in the Disputes System shall be in compliance to the guidelines of Regulation E and Regulation Z.,L\nPROMISE-0177,4,The Disputes application must maintain a detailed history of every action that a user takes on a dispute case.  This ensures a complete audit trail if questions arise later on with regard to a particular dispute case.,L\nPROMISE-0178,4,All actions that modify an existing dispute case must be recorded in the case history.,L\nPROMISE-0179,4,The Disputes System must be accessible by both internal and external users.,F\nPROMISE-0180,4,The Disputes System must prevent users from accessing any dispute cases that do not belong to their cardholder base.,F\nPROMISE-0181,4,The Disputes System will facilitate direct data entry of a dispute case via a user interface that supports real time responses to the users.,F\nPROMISE-0182,4,The Disputes System must provide different levels of access with regard to disputes case initiation and follow-up actions.,F\nPROMISE-0183,4,The Disputes System shall provide view access capability for authorized users of the application.,F\nPROMISE-0184,4,The Disputes System shall provide update access capability for authorized users of the application.,F\nPROMISE-0185,4,The Disputes System must allow the users to select disputable transactions (based on the age of the transaction) from a user interface and initiate a dispute (ticket retrieval request or chargeback notification) on the selected transaction.,F\nPROMISE-0186,4,The Disputes System must provide the user the ability to initiate a single dispute case on multiple transactions that belong to a single merchant.,F\nPROMISE-0187,4,The Disputes System will provide the user the ability to create or initiate a ticket retrieval request.   As part of ticket retrieval creation process the system must prompt the user to enter all the required information to create the ticket retrieval request.  The ticket retrieval request is a document that is sent to merchant inquiring the validity of a transaction.,F\nPROMISE-0188,4,The Disputes System must allow the user to create three unique types of ticket retrieval requests.  The three types of ticket retrieval requests are (1) Request for original receipt (2) Request for a copy of the receipt or (3) Request for a portfolio. A portfolio consists of documentation that would provide proof of a purchase such as the documentation that is received from a car rental agency that is more than a sales receipt.,F\nPROMISE-0189,4,The Disputes System must prevent external users from requesting original receipts. Requests for original receipts are restricted to internal users.,F\nPROMISE-0190,4, The Disputes System must provide a confirmation to the user upon the creation of ticket retrieval request that contains the following information; the dispute case number  the type of retrieval requested (copy  original or portfolio)  and the date that the merchant response is due.,F\nPROMISE-0191,4,The Disputes System shall allow the user to create or initiate a chargeback request.  The system must prompt the user to enter the required information to initiate a chargeback request. The chargeback request results in a printed chargeback notification that is sent to merchant indicating that their account will be charged back if no proof is provided that the disputed transaction is valid.,F\nPROMISE-0192,4, The Disputes System must provide a confirmation to the user upon the creation of a chargeback request. The confirmation must contain the following information; the dispute case number  the type of chargeback requested (pending or immediate)  and the date that the merchant response is due.,F\nPROMISE-0193,4,The Disputes System shall prevent the creation of duplicate dispute requests 100% of the time.,F\nPROMISE-0194,4, The Disputes System shall accurately provide a list of possible reason codes for a given transaction.  The list of possible reason codes must be derived based on the type of transaction and the industry that the merchant belongs to  according to the business rules defined for the Disputes System.,F\nPROMISE-0195,4,The Disputes System must provide the ability to view existing dispute cases.,F\nPROMISE-0196,4, The Disputes System must provide Search functionality.  The search method must include the ability to search by (1) the dispute case number  (2) the merchant account number  (3) the cardmember account number and (4) the issuer number.  In addition to the above criteria  the search functionality must further allow the user to limit the results of the search by a date range  the type of dispute (ticket retrieval request or chargeback notification)  the case status (open  closed or all) and the dispute reason code.,F\nPROMISE-0197,4,A list of all disputes cases that meet the entered search criteria must be provided to the user.  The list of cases must differentiate the type and/or status of the case by color -coding the line item in the list.  For example all open cases will be yellow and all closed cases will be gray.,F\nPROMISE-0198,4,The Disputes System shall provide the users the ability to view the history of all activities that have occurred on the dispute case.,F\nPROMISE-0199,4,The Dispute System shall provide the users with the ability to follow up on a dispute case via follow-up actions.  The list of available follow up actions for a given dispute case must be filtered by the status of the case and the access level of the user.,F\nPROMISE-0200,4, The Disputes System shall record the name of the user and the date for any activity that creates or modifies the disputes case in the system.  A detailed history of the actions taken on the case  including the date and the user that performed the action must be maintained for auditing purposes.,F\nPROMISE-0201,4, For any systematic (non-user initiated) action that occurs on a case such as the disputes aging process  a case activity will be generated.  Upon the next logon the user that initiated the dispute case must be notified that a systematic action has occurred on the dispute case.,F\nPROMISE-0202,4,All letter requests must be formatted according to guidelines specified by the Print Letter Utility system.,F\nPROMISE-0203,4,Any disputes cases that have been closed for over 6 months must be purged from the online disputes database.,F\nPROMISE-0204,5,The product must support Internet Explorer 5.5 and above.,O\nPROMISE-0205,5,The product must support Mozilla Firefox 1.0 and above.,O\nPROMISE-0206,5,The product shall interface with the Choice Parts System.  This provides the feed of recycled parts data.,O\nPROMISE-0207,5,The product shall run on the existing hardware for all environments.,O\nPROMISE-0208,5,The product shall adhere to the corporate Architecture guidelines,O\nPROMISE-0209,5,The product shall comply with corporate User Interface Guidelines,LF\nPROMISE-0210,5,The product shall comply with corporate color scheme,LF\nPROMISE-0211,5,The appearance of the product shall appear professional,LF\nPROMISE-0212,5,The product shall be easy to use by Adjusters and Collision Estimators.  95% of Adjusters and Collision Estimators shall find the product easy to use.,US\nPROMISE-0213,5,The product shall increase productivity of Collision Estimators. 80% of the Collision Estimators shall agree their productivity has increase within 1 month of using the product.,US\nPROMISE-0214,5,Users shall feel satisfied using the product.  85% of all users will be satisfied with the product.,US\nPROMISE-0215,5,The product shall be easy to learn by Adjusters and Collision Estimators.  The product shall be learned with two days onsite training,US\nPROMISE-0216,5,The user shall easily locate instructions while using the product.  User help can be found within 90% of the system.,US\nPROMISE-0217,5,The search for recycled parts shall take no longer than 15 seconds.  The search results shall be returned in under 15 seconds.,PE\nPROMISE-0218,5,The search for the preferred repair facility shall take no longer than 8 seconds.  The preferred repair facility is returned within 8 seconds,PE\nPROMISE-0219,5,The recycled parts audit report shall be returned to the user within 10 seconds.  The audit report shall be returned within 10 seconds.,PE\nPROMISE-0220,5,The preferred repair facility ratings shall be saved within 5 seconds. The save shall occur within 5 seconds.,PE\nPROMISE-0221,5,The product shall adhere to the corporate online availability schedule.  The application is brought down only within 98% of the scheduled outages per the availability schedule.,A\nPROMISE-0222,5,The product shall achieve a 98% uptime. The product shall not fail more than 2% of the available online time.,A\nPROMISE-0223,5,The product shall be able to handle up to 1500 simultaneous users. The system shall perform with 1500 users as a time.,SC\nPROMISE-0224,5, The average number of recycled parts records per day shall be 50 000.,SC\nPROMISE-0225,5, The product shall be able to handle 10 000 concurrent users within 2 years of the initial launch.,SC\nPROMISE-0226,5, For estimators  the product shall be able to be operated in a repair facility during dirty and noisy conditions.,O\nPROMISE-0227,5,The product shall interface with the ChoiceParts system once per day at approximately 1:00 AM.,O\nPROMISE-0228,5,The maintenance of the product will fall within the established corporate maintenance windows.,O\nPROMISE-0229,5,The product shall be supported using the corporate support center.,O\nPROMISE-0230,5,Product installations and upgrades shall be handled by the middleware technology team within established launch time frames.,O\nPROMISE-0231,5,Only adjusters can request recycled parts audit reports. No users without an adjuster role shall request recycled parts audits.,SE\nPROMISE-0232,5,Only adjusters with a supervisor role can update preferred repair facility ratings. Users without the supervisor role cannot access the repair facility ratings.,SE\nPROMISE-0233,5,Only collision estimators shall search for recycled parts.  Users without the collision estimator role shall not have access to the search for recycled parts.,SE\nPROMISE-0234,5,Only valid data shall be entered into the system.  No invalid data shall be entered into the system.,SE\nPROMISE-0235,5,One insurance company shall not be able to view another insurance companys claim data.,SE\nPROMISE-0236,5,The product shall free of computer viruses.,SE\nPROMISE-0237,5,The system shall prevent malicious attacks including denial of service.,SE\nPROMISE-0238,5,The product must comply with Sarbanes-Oxley.,L\nPROMISE-0239,5,The product shall comply with the estimatics laws relating to recycled parts usage.,L\nPROMISE-0240,5,The product shall comply with insurance regulations regarding claims processing.,L\nPROMISE-0241,5, The user shall search for the preferred repair facility using vehicle  vehicle location  and radius in miles,F\nPROMISE-0242,5, The vehicle data shall include vehicle year  make  and model,F\nPROMISE-0243,5, The vehicle location shall include street address  city  state  and zipcode,F\nPROMISE-0244,5, The system shall resolve the zipcode for the street address  city  and state if zipcode is unknown.,F\nPROMISE-0245,5,The search radius shall be between 1 and 30 miles,F\nPROMISE-0246,5,The system shall locate the preferred repair facility with the highest ratings for the input criteria,F\nPROMISE-0247,5,The adjuster shall enter the preferred repair facility on the estimate assignment.,F\nPROMISE-0248,5,The adjuster shall be able to override the preferred repair facitlity on the estimate assignment.,F\nPROMISE-0249,5,The preferred repair facility shall be returned to the user.,F\nPROMISE-0250,5,The system shall return a list of repair facilities within the radius if the preferred repair facility cannot be determined.,F\nPROMISE-0251,5,The estimator shall search for available recycled parts using damaged vehicle parts information.,F\nPROMISE-0252,5,The estimator shall search for available recycled parts using a list of preferred parts suppliers.,F\nPROMISE-0253,5,The system shall search for available recycled parts for the supplied vehicle parts and suppliers.,F\nPROMISE-0254,5,The estimator shall search by zipcode with a radius of 30 miles if no parts are found for the preferred parts suppliers,F\nPROMISE-0255,5,The system shall retain the available recycled parts and suppliers returned from the search.,F\nPROMISE-0256,5,The available recycled parts information and their suppliers shall be returned to the user.,F\nPROMISE-0257,5,The estimator shall apply selected recycled parts to the collision estimate.,F\nPROMISE-0258,5,The estimator shall not apply recycled parts to the collision estimate if no available parts are returned.,F\nPROMISE-0259,5,The adjuster shall review the collision estimate.,F\nPROMISE-0260,5,The adjuster shall request a recycled parts audit of the collision estimate.,F\nPROMISE-0261,5,The recycled parts search results provided to the estimator shall be retrieved by the system.,F\nPROMISE-0262,5,The system shall generate an audit report based on the available recycled parts and the collision estimate.,F\nPROMISE-0263,5,The audit report shall include the total number of recycled parts used in the estimate.,F\nPROMISE-0264,5,The audit report shall include the number of available recycled parts from the original search results.,F\nPROMISE-0265,5, The audit report shall include the percentage of available recycled parts used in the estimate ,F\nPROMISE-0266,5,The audit report shall include the total score of the audit which sums the indivual line items.,F\nPROMISE-0267,5,The audit report shall be displayed to the user.,F\nPROMISE-0268,5,The audit report shall be available to other adjusters at later points in time.,F\nPROMISE-0269,5,The user shall select to view the preferred repair facility ratings.,F\nPROMISE-0270,5,The current repair facility ratings shall be displayed to the user.,F\nPROMISE-0271,5,The system will display a blank set of ratings if there are not ratings yet defined.,F\nPROMISE-0272,5,The user shall enter new ratings based on the audit report,F\nPROMISE-0273,5,The ratings shall be from a scale of 1-10,F\nPROMISE-0274,5,The ratings shall include categories for attempted use of recycled parts and actual use of recycled parts,F\nPROMISE-0275,5,The user shall select to save the preferred repair facility ratings.,F\nPROMISE-0276,5,The preferred repair facility ratings shall be saved by the system.,F\nPROMISE-0277,6, The product must work with most database management systems (DBMS) on the market  whether the DBMS is colocated with the product on the same machine or is located on a different machine on the computer network.,O\nPROMISE-0278,6,The product will function alongside server software on any operating system where the Java runtime can be installed.,O\nPROMISE-0279,6,The product will require collaboration with a database management system (DBMS).  The DBMS may be located on the same machine as the product or on a separate machine residing on the same computer network.,O\nPROMISE-0280,6, The product must make use of web/application server technology.  Open source examples include Apache web server  Tomcat  and the JBoss application server.,O\nPROMISE-0281,6, A database management system such as Oracle  DB2  MySql or HSQL will need to be integrated with the product.,O\nPROMISE-0282,6, The product’s Look and Feel shall be able to incorporate aspects of the customer’s organization such as logo  branding  and identity,LF\nPROMISE-0283,6,The product shall have a conservative and professional appearance.,LF\nPROMISE-0284,6,The product shall make the users want to use it. 80% of the users surveyed report they are regularly using the product after the first 2 weeks post-launch.,US\nPROMISE-0285,6,The product shall be easy to use. 90% of users will be able to successfully reserve a conference room within 5 minutes of product use.,US\nPROMISE-0286,6,The product shall give users feedback when necessary. 80% of the users surveyed report that the product accurately confirms their actions.,US\nPROMISE-0287,6,The product shall allow the user to select a chosen language from one of the target market countries.,US\nPROMISE-0288,6,The product shall allow for customization of start page and views preferences.,US\nPROMISE-0289,6, An employee will be able to successfully use the product within a few minutes.  After informally navigating the product for less than 15 minutes  users shall be able to successfully setup meetings and reserve conference rooms.,US\nPROMISE-0290,6,The product shall use a standard navigation menu familiar to most web users.,US\nPROMISE-0291,6,The product shall allow for intuitive searching of available conference rooms.,US\nPROMISE-0292,6,The product shall conform to the Americans with Disabilities Act.,US\nPROMISE-0293,6,The response shall be fast enough to avoid interrupting the user’s flow of thought.  90% of tasks shall complete within 5 seconds.  98% of tasks shall complete within 8 seconds.,PE\nPROMISE-0294,6, Aside from server failure  the software product shall achieve 99.99% up time.,A\nPROMISE-0295,6,The product shall create an exception log of problems encountered within the product for transmission to our company for analysis and resolution.,FT\nPROMISE-0296,6, The product shall be capable of handling up to 1 000 concurrent requests.  This number will increase to 2 000 by Release 2.  The concurrency capacity must be able to handle peak scheduling times such as early morning and late afternoon hours.,SC\nPROMISE-0297,6, The product shall be able to process 10 000 transactions per hour within two years of its launch.  This number will increase to 20 000 by Release 2.,SC\nPROMISE-0298,6,The product shall be expected to operate for at least 5 years for each customer installation.,SC\nPROMISE-0299,6,The product shall be used in office environments.,O\nPROMISE-0300,6,The product must be able to interface with any HTML browser.  The product shall transmit data between the user and the product without problems. The product shall display HTML properly in 80% of all HTML browsers tested (minimum is to test 8 browsers).,O\nPROMISE-0301,6,The product must be able to interface with various database management systems. The product shall communicate successfully with the database management system on 100% of all transactions.,O\nPROMISE-0302,6,The product must be able to interface with various email servers. The product shall be able to send email.,O\nPROMISE-0303,6,The product shall be available for distribution via the Internet as a binary or ZIP file.,O\nPROMISE-0304,6,The product shall be available for distribution as a packaged CD.,O\nPROMISE-0305,6, The product will be available for licensing as a one-server  two-five servers  or five-or-more servers license.,O\nPROMISE-0306,6,A new user must be able to be added to the system within 10 minutes.  90% of new users are able to log into the system within 10 minutes.,US\nPROMISE-0307,6,The product must be designed using Design Patterns and coding best practices.  90% of maintenance software developers are able to integrate new functionality into the product with 2 working days.,MN\nPROMISE-0308,6,The product must be highly configurable for use with various database management systems for the end users.  80% of end users are able to integrate new database management systems with the product without changing the product’s software code.,MN\nPROMISE-0309,6,Maintenance releases will be offered to customers once a year.  The releases shall take no longer than 5 minutes for 80% of the customer base to install.,MN\nPROMISE-0310,6,Every registered user will have access to the product’s support site via the Internet.  70% of registered users shall find a solution to their problem within 5 minutes of using the support site.,US\nPROMISE-0311,6,The customer shall be able to easily integrate new building maps with the product throughout the product’s lifecycle.  Integration of new maps with the product shall be possible with little to no support from product support staff.,MN\nPROMISE-0312,6, The product shall be translated into foreign languages other than the target market countries’ languages in future releases.  For each emerging target market  it shall take no more than 5 days to configure the product for that market’s language.,US\nPROMISE-0313,6, The product is expected to integrate with multiple database management systems.  The product will operate with Oracle  SQL Server  DB2  MySQL  HSQL  and MS Access.,O\nPROMISE-0314,6,The product shall be able to be installed in any operating environment within 2 days.,O\nPROMISE-0315,6,The product shall ensure that only company employees or external users with company-approved user IDs may have product access.  100% of all logons shall be by either company employees or external users with company-approved user IDs.,SE\nPROMISE-0316,6,Only managers are able to perform search queries for reservations by user.  100% of all search queries for reservations by user shall be from logons of only management logons.,SE\nPROMISE-0317,6,The product shall ensure that the database’s data corresponds to the data entered by the user.  Each conference room reservation in the system will contain the same information as was entered by the user.,SE\nPROMISE-0318,6,The product shall prevent the input of malicious data.  The product and/or adjacent hardware/software systems’ data shall remain 100% uncorrupted each time malicious data is input into the product.,SE\nPROMISE-0319,6,The product and/or dependent databases/filesystems shall remain operational as a result of the input of malicious data.  The product and/or dependent databases/filesystems shall not crash 100% of the time that malicious data is input into the product.,FT\nPROMISE-0320,6,The product shall have the ability to receive automatic software updates as new threats emerge.  100% of customers will be able to receive automatic software updates transmitted to the installed product.,SE\nPROMISE-0321,6,The language of the product shall accommodate all of the target market countries. 90% of survey respondents from each target market country will find that the language of the product is acceptable.,SC\nPROMISE-0322,6,The product shall be able to display calendar dates and times according to the user’s locale.  90% of survey respondents from each target market country will find that the date and time formatting of the product is correct.,US\nPROMISE-0323,6,The product must be developed with the J2SE/J2EE programming language libraries,O\nPROMISE-0324,6,The product shall record meeting entries.,F\nPROMISE-0325,6,The product will notify employees of meeting invitations.,F\nPROMISE-0326,6,The product shall have the ability to send meeting reminders to employees.,F\nPROMISE-0327,6,The product shall assign the organizers contact information to each meeting they create.,F\nPROMISE-0328,6,The product will record meeting acknowledgements.,F\nPROMISE-0329,6,The product shall store new conference rooms.,F\nPROMISE-0330,6,The product shall update existing conference rooms.,F\nPROMISE-0331,6,The product will be able to delete conference rooms.,F\nPROMISE-0332,6,The product shall be able to store new room equipment.,F\nPROMISE-0333,6,The product will update existing room equipment.,F\nPROMISE-0334,6,The product shall be able to delete room equipment.,F\nPROMISE-0335,6,The product shall allow an organizer to invite other employees to meetings.,F\nPROMISE-0336,6, Each time a conference room is reserved  the conference room schedule shall be updated to reflect the time and date of the reservation.,F\nPROMISE-0337,6,The product shall record the transportation status of equipment reserved.,F\nPROMISE-0338,6,The product shall display a map of the company building showing conference room locations.,F\nPROMISE-0339,6,The product shall record updated meeting agendas.,F\nPROMISE-0340,6,The product shall send a meeting confirmation to the meeting organizer.,F\nPROMISE-0341,6,The product shall display room equipment according to search parameters.,F\nPROMISE-0342,6,The product shall display conference rooms according to search parameters.,F\nPROMISE-0343,6,The product shall record different meeting types.,F\nPROMISE-0344,6,The product shall record all the equipment that has been reserved.,F\nPROMISE-0345,6,The product shall notify building personnel of equipment transport requests.,F\nPROMISE-0346,6,The product will allow priviledged users to view meeting schedules in multiple reporting views.,F\nPROMISE-0347,6,The product shall be able to send meeting notifications via different kinds of end-user specified methods.,F\nPROMISE-0348,6,The product shall have a customizable Look and Feel.,F\nPROMISE-0349,6,The product shall have an intuitive user interface.,F\nPROMISE-0350,6,The product will display an available status for unreserved conference rooms.,F\nPROMISE-0351,7,Recreation of Inventory Quantity Adjustment for the period of previous 30 days shall take no more than 30min.,PE\nPROMISE-0352,7,The System shall allow for a minimum of  6 users to work at the same time.,PE\nPROMISE-0353,7,The System shall utilize currently owned computer equipment.,O\nPROMISE-0354,7,The System shall be able to automatically backup business data and restore them on demand.,O\nPROMISE-0355,7,The System shall maintain consistent security.  Any changes in user passwords and/or permissions should take affect in all parts of the System.,SE\nPROMISE-0356,7,The System shall granularly control access rights to all major subsystems.  It should be possible to assign access rights to all subsystems independently.,SE\nPROMISE-0357,7,The System shall meet all applicable accounting standards.  The final version of the System must successfully pass independent audit performed by a certified auditor.,L\nPROMISE-0358,7,The System shall not require additional third party licenses resulting in royalty fees.,O\nPROMISE-0359,7,The System shall allow generation of Inventory Quantity Adjustment documents on demand.,F\nPROMISE-0360,7,The System shall allow on demand generation of all Inventory Quantity Adjustment documents since certain point of time.,F\nPROMISE-0361,7,The IQA subsystem shall take into account current inventory status and adjust inventory of substitutionary ingredients accordingly.,F\nPROMISE-0362,7, The System shall allow entering  storing and modifying product formulas.,F\nPROMISE-0363,7, Product formulas shall allow use of semi-products  which have own formulas.,F\nPROMISE-0364,7,Product formula ingredients shall allow defining substitutionary ingredients.,F\nPROMISE-0365,7,Product formula substitutionary ingredients shall be entered automatically by the PFE subsystem.,F\nPROMISE-0366,7,Formula substitutionary ingredients shall be kept in persisted order,F\nPROMISE-0367,7, Product shall allow entering  storing and modifying product menus used by POS terminals.,F\nPROMISE-0368,7,PME Subsystem shall allow building product menus based on the list of products kept in the accounting system.,F\nPROMISE-0369,7,PME Subsystem shall allow keeping submenus within menus and products within submenus in a managed and persisted order.,F\nPROMISE-0370,7,PME Subsystem shall allow for a product to exist in more than one submenu,F\nPROMISE-0371,7,The RMS System shall have Windows XP look and feel and use GUI futures making its use intuitive.,F\nPROMISE-0372,7,The System shall generate Inventory Quantity Adjustment document based on product daily sales data and product formulas.,F\nPROMISE-0373,7,The System shall generate Inventory Quantity Adjustment document automatically when daily Product Sales data is available.,F\nPROMISE-0374,8,Administrator shall be able to activate a pre-paid card via the Administration section in under 5 seconds.,PE\nPROMISE-0375,8, Customer shall be able to check the status of their prepaid card  by entering in the PIN number  in under 5 seconds.,PE\nPROMISE-0376,8,System shall let customers register on the website as a “pay as you go” user in under 5 minutes.,PE\nPROMISE-0377,8,System shall let customers register on the website as a “unlimited movie subscriber” user in under 10 minutes.,PE\nPROMISE-0378,8,System shall let existing customers log into the website with their email address and password in under 5 seconds.,PE\nPROMISE-0379,8, System shall let customers cancel their account  process shall take no longer than 2 minutes.  Database will mark the account as inactive  and user will not be able to log into the website.,PE\nPROMISE-0380,8,System shall let administrator de-activate a customer account in under 1 minute. Customer will no longer be able to access the website.,PE\nPROMISE-0381,8,System shall let administrator add/remove movies on the website in under 5 minutes.  Entered movie information will be stored in the database and will now be available on the website.,PE\nPROMISE-0382,8,System shall let the administrator add/remove categories on the website in under 5 minutes.,PE\nPROMISE-0383,8,System shall let Izogn Manager access sales and usage statistics in under 2 minutes.,PE\nPROMISE-0384,8,System shall let administrators generate unique pre-paid pins in under 2 minutes and generate an Excel file with the generated PINs.,PE\nPROMISE-0385,8,The website shall be attractive to all audiences.  The website shall appear to be fun and the colors should be bright and vibrant.,LF\nPROMISE-0386,8, The website should have an African feel  but should not alienate non-Africans.  The website should use animation on pages which are describing the services  to grab the users attention and encourage them to sign up.,LF\nPROMISE-0387,8, The website design should be modern  clean  and concise.,LF\nPROMISE-0388,8,The product should be able to be used by 90% of novice users on the Internet.,US\nPROMISE-0389,8, Users should only have to navigate through a maximum of 4 pages  to create an account with the website.,US\nPROMISE-0390,8,Users should be able to access their streaming movies in under 2 clicks after logging into the website.,US\nPROMISE-0391,8,An anonymous survey will show that 70% of customers are comfortable purchasing streaming movies after 2 weeks using the website.,US\nPROMISE-0392,8,An anonymous survey will show that 95% of customers are comfortable using and purchasing pre-paid cards after a month of using the system.,US\nPROMISE-0393,8,The product shall retain the customer’s buying preferences and information.,US\nPROMISE-0394,8,The product shall be easy for customers with novice skills on the  Internet to purchase and watch streaming movies.  Customers should be able to purchase and watch a pre-determined movie in under 3 minutes.,US\nPROMISE-0395,8,The product shall use symbols and words that are naturally understandable by users familiar with websites,US\nPROMISE-0396,8,The website will use standard Internet conventions when interacting with the user.,US\nPROMISE-0397,8, Any interface between a user and the automated system shall have a maximum response time of 5 seconds  unless noted by an exception below.,PE\nPROMISE-0398,8, When purchasing a streaming movie or pre-paid card via credit card  the processing time should have a maximum response time of 15 seconds.,PE\nPROMISE-0399,8, When streaming a movie  the buffering time should take no longer than 10 seconds (plus any latency on the user’s connection.),PE\nPROMISE-0400,8, While streaming a movie  buffering should only occur if there is latency in the user’s connection.,PE\nPROMISE-0401,8, All credit card information will be secured on the server  and only accessible by authorized Izogn administrators.  Information will be encrypted in the database.,SE\nPROMISE-0403,8,The website shall achieve 99.5% up time.,A\nPROMISE-0404,8, All movies shall be streamed on demand  at any time of the day.,A\nPROMISE-0405,8,The website shall continue to operate if the streaming server goes down.,FT\nPROMISE-0406,8,The website shall continue to operate if the payment gateway goes down.,FT\nPROMISE-0407,8,The streaming server will support 50 simultaneous streaming connections.,SC\nPROMISE-0408,8,The streaming server will support a maximum of 700 movies; new servers can be added to support more movies.,SC\nPROMISE-0409,8, The server will support a maximum of 1 000 simultaneous users.,SC\nPROMISE-0410,8, The website shall be capable of supporting 100 000 customers.,SC\nPROMISE-0411,8,The product shall be expected to operate within the maximum maintenance budget for a minimum of 2 years.,MN\nPROMISE-0412,8,All aspects of the website shall be accessed by a web browser over the Internet.,O\nPROMISE-0413,8,The same hardware supporting the current website will be used to support the new system.,O\nPROMISE-0414,8, We must be able to interface with the following browsers: IE 5.X  6.0  Netscape 6.x  7.x  8.x and Firefox 1.0.,O\nPROMISE-0415,8,Website must interface with the CyberSource API to process credit card transactions.,O\nPROMISE-0416,8,Website must be fully operational with MSN TV2.,O\nPROMISE-0417,8,Movies shall be encoded and streamed via Windows Media Player 9.0.,O\nPROMISE-0418,8,Movies shall be encoded for 56k and 256k streams.,O\nPROMISE-0419,8,Promotional updates to the website should take a day to update.,MN\nPROMISE-0420,8,Izogn Administrator must be able to update the category listings on the website within 2 minutes.,PE\nPROMISE-0421,8,Izogn Administrator must be able to add new products to the website within 2 minutes.,PE\nPROMISE-0422,8,The system will provide a 24 hour 800 toll free number for support to its customers.,A\nPROMISE-0423,8,The website will provide a help section with answers and solutions to common problems.,US\nPROMISE-0424,8, Velkat staff will be available for phone support between 7am-7pm  M-F to provide technical support to the Izogn administrators.,A\nPROMISE-0425,8,The product shall be able to be installed in the specified environment within 1 working day.,O\nPROMISE-0427,8, A customer can only stream a movie if they purchased it  and are within the 2 day viewing period.,SE\nPROMISE-0428,8,Only authorized personnel can access customer records in the database.,SE\nPROMISE-0429,8,Only administrators can activate a pre-paid card.,SE\nPROMISE-0430,8,Only authorized personnel can access sales information.,SE\nPROMISE-0431,8,The website shall prevent its data from being corrupted when incorrect data is entered in.,SE\nPROMISE-0432,8, The website shall protect itself from intentional abuse  and notify the administrator at all occurrences.,SE\nPROMISE-0433,8,Data integrity scripts will be run on a weekly basis to verify the integrity of the database.,SE\nPROMISE-0434,8, The website shall make its user aware of its information practices before collection data from them via a Privacy Policy  accessible on all pages of the website.,SE\nPROMISE-0435,8,The system shall notify customers of changes to its information policy.,SE\nPROMISE-0436,8,The system shall reveal private information only in compliance with the organization’s information policy.,SE\nPROMISE-0437,8, All customer information will be stored on a secure database  accessible only to authorized personnel.,SE\nPROMISE-0438,8,All credit card information will be encrypted in the database.,SE\nPROMISE-0439,8,The streaming server will have anti-virus software to prevent infection of malicious viruses.,SE\nPROMISE-0440,8,The website should cater to all tribes in Nigeria.,LF\nPROMISE-0441,8,The website shall not be offensive to religious or ethnic groups.,LF\nPROMISE-0442,8, The website should appeal to all Africans  not only Nigerians.,LF\nPROMISE-0443,8,The website should use Flash to provide animation on the home page.,O\nPROMISE-0444,8,IzognMovies shall meet the licensing requirements set forth by the appropriate organizations in order to stream African movies.,L\nPROMISE-0445,8,The website will comply with W3C standards.,L\nPROMISE-0446,8,The PHP code will comply with PEAR standards.,L\nPROMISE-0447,8, The website will allow customers to search for movies by title  actor  or director.,F\nPROMISE-0448,8, viewing a movie details  the website will display the movies description  actor  and director entered in by the administrator.,F\nPROMISE-0449,8, Website shall request  and authorize credit card payment before allowing a user to stream a movie.,F\nPROMISE-0450,8, shall be able to stream purchased movie via Windows Media Player  for up to 2 days aftere purchase date.,F\nPROMISE-0451,8,Website shall allow customers to browse movies by categories entered in by izogn administrator.,F\nPROMISE-0452,8,Website shall allow customers to view reviews of selected movies by other customers.,F\nPROMISE-0453,8,Website shall allow customers to add their own movie review for a selected movie.,F\nPROMISE-0454,8,Website shall allow the Izogn administrator to approve a review posted by a customer.,F\nPROMISE-0455,8,System will allow customers to pay for a streaming movie with a pre-paid card.,F\nPROMISE-0456,8, Website shall allow customers to purchase pre-paid cards of $5  $10  or $20 value either by credit card or mail-in payment option.,F\nPROMISE-0457,8, System shall automatically update the main page of the website every Friday  and show the 4 latest movies that have been added to the website.,F\nPROMISE-0458,8,System shall allow users to update their billing and contact information via the “My Account” section of the website.,F\nPROMISE-0459,8, Pre-paid cards shall be distributed via email  or via a printed paper card.,F\nPROMISE-0461,8, Movies can only be streamed if the customer has purchased a movie  and is within the 2-day time period to stream the movie.,F\nPROMISE-0462,8,Website must be able to support free trial periods with various parameters set by the Izogn Manager.,F\nPROMISE-0463,8, All streaming movie sales will be logged in the database  and accessible by the administrator.,F\nPROMISE-0464,8, All registered customer information shall be stored and maintained in the database  even after the customer cancels their account.,F\nPROMISE-0465,8, Daily usage statistics should be logged  and accessible by the administrator.,F\nPROMISE-0466,8,The owner shall have free access to all of the streaming movies.,F\nPROMISE-0467,9,Total lead cycle time must be completed in 5 minutes for 90% of lead volume between the business hours of 8AM CST to 8PM CST; then 3 minutes for 99% of lead volume during off business hours.,PE\nPROMISE-0468,9,No lead evaluation will complete in a time period longer than 25 seconds for 97% of all leads processed.,PE\nPROMISE-0469,9,For leads that process longer than 25 seconds the system will record the event and duration,PE\nPROMISE-0470,9,The process must fit within the processing time conditions set forth by the Auto Rep Assignment process in which a lead is assigned within 3 minutes of the total life cycle.,PE\nPROMISE-0471,9,The lead washing process will operate within the established physical web service structure.,O\nPROMISE-0472,9, The washing process must use a non binding  neutral data format  such as xml  when receiving and processing lead data.,O\nPROMISE-0473,9,Defect reports will be available to technical units on a 24x7 basis.,A\nPROMISE-0474,9,Washing parameters will be updated during scheduled maintenance hours.,MN\nPROMISE-0475,9,The vendor will verify the invalid lead.,F\nPROMISE-0476,9,Vendor will submit a credit validation record to be applied to the enterprise billing system.,F\nPROMISE-0477,9,If a lead is scored to the high range the system will supply the lead data and score to the enterprise system.,F\nPROMISE-0478,9,The leads washing functionality will validate all leads received by the web service for valid data,F\nPROMISE-0479,9,The leads washing functionality will insert all leads captured by the web service.,F\nPROMISE-0480,9,If the leads score falls within the high average score then it will be process by the ARA functionality.,F\nPROMISE-0481,9,If the leads score falls within the medium average then it will be set for manual verification by an Enrollment Coordinator through the eleads system.,F\nPROMISE-0482,9,If the leads score falls within the low average then it will be returned to the supplying vendor,F\nPROMISE-0483,9,The leads washing functionality will compile both contact and academic scoring information for a total score.,F\nPROMISE-0484,9,The leads washing functionality will use the parameters supplied by corporate and schools to score part of the lead.,F\nPROMISE-0485,9,The leads washing functionality will return the lead data supplied to the vendor along with the reason of rejection.,F\nPROMISE-0486,9,The leads washing functionality will have an interface in which lead data parameters can be maintained.,F\nPROMISE-0487,9,The leads washing functionality will verify the users authentication and rights to make parameter updates.,F\nPROMISE-0488,9,The leads washing functionality will store any potential lead duplicates returned by the enterprise system.,F\nPROMISE-0489,9,Prior to submitting the lead to the enterprise system the leads washing functionality will assign the lead to an admissions rep.,F\nPROMISE-0490,9,The leads washing functionality shall receive a period list of available reps from the enterprise system.,F\nPROMISE-0491,10,The product shall simulate the look of ships at sea.,LF\nPROMISE-0492,10,The product shall display the grids within a circle as a view from a periscope.,LF\nPROMISE-0493,10,The product shall display each type of ship in a grid using an image of the specific type of ship.,LF\nPROMISE-0494,10, When the offensive player takes a shot  the product shall simulate the sound of a ship at sea.,LF\nPROMISE-0495,10, When a shot is marked as a miss  the product shall simulate the sound of a shot missing its target.,LF\nPROMISE-0496,10, When a shot is marked as a hit  the product shall simulate the sound of a shot hitting its target.,LF\nPROMISE-0497,10, When a ship is sunk  the product shall simulate the sound of a sinking ship.,LF\nPROMISE-0498,10, The product shall be available 24 hours per day  seven days per week.,A\nPROMISE-0499,10,The product shall able to handle a large number of games simultaneously.,PE\nPROMISE-0500,10,The response time shall be fast enough to maintain the flow of the game.  The response time shall be no more than 2 seconds for 95% of responses and no more than 4% for the remaining responses.,PE\nPROMISE-0501,10,The product shall protect the identity of the players. The product shall provide players no access to information that might reveal the identity of another player.,PE\nPROMISE-0502,10,The product shall be intuitive and self-explanatory.  Nine out ten users shall be able to successfully complete a game with no training.,US\nPROMISE-0503,10, The product shall be available to players who understand English  Spanish and French.  The product shall allow the user to select the language upon accessing the product.,US\nPROMISE-0504,10,The product shall provide players no access to information that might reveal the identity of another player,SE\nPROMISE-0505,10,The response time shall be fast enough to maintain the flow of the game. The response time shall be no more than 2 seconds for 95% of responses and no more than 4% for the remaining responses.,PE\nPROMISE-0506,10,The product shall maintain a list of players.,F\nPROMISE-0507,10,The product shall allow a user to identify himself or herself as a player.,F\nPROMISE-0508,10, Once a game has been initiated  the product shall make the players assigned to the game unavailable for selection from the list of players.,F\nPROMISE-0509,10,The product shall allow a user to remove himself or herself from the list of players at any time.,F\nPROMISE-0510,10,The product shall make inactive players unavailable for selection from the list of players.,F\nPROMISE-0511,10,The product shall remove players who have been inactive for 24 hours from the list of players.,F\nPROMISE-0512,10,The product shall allow a player to initiate a game by selecting another available player from the list of players.,F\nPROMISE-0513,10, Once a game is initiated  the product shall display a defensive grid to each player,F\nPROMISE-0514,10, Once a game is initiated  the product shall display an offensive grid to each player.,F\nPROMISE-0515,10,The product shall prevent a player from viewing the offensive or defensive grids of the other player.,F\nPROMISE-0516,10, Once a game is initiated  the product shall allow each player to position their 5 ships on their respective defensive grids.,F\nPROMISE-0517,10,The product shall allow a player to position ships horizontally on their grid.,F\nPROMISE-0518,10,The product shall allow the player to position ships vertically on their grid.,F\nPROMISE-0519,10,The product shall allow the player to position ships diagonally on their grid,F\nPROMISE-0520,10,The product shall prevent the player from overlapping ships on their grid.,F\nPROMISE-0521,10, When both players have positioned their 5 ships on their defensive grids  the product shall define the status of each player for the first turn.,F\nPROMISE-0522,10, For each turn  the product shall identify one player as offensive status and one player as defensive status.,F\nPROMISE-0523,10, The product shall maintain the status of each player for a turn  either offensive or defensive.,F\nPROMISE-0524,10, At the start of each turn  the product shall notify each player of his or her status.,F\nPROMISE-0525,10, For each turn  the product shall allow the offensive player to define a shot.,F\nPROMISE-0526,10, For each shot  the product shall examine the defensive grid of the defensive player to determine whether the shot was a hit or miss.,F\nPROMISE-0527,10, For each shot  the product shall notify the players whether the shot was a hit or miss.,F\nPROMISE-0528,10, For each shot  the product shall mark the shot as a hit or miss on the defensive grid of the defensive player.,F\nPROMISE-0529,10, For each shot  the product shall mark the shot as a hit or miss on the offensive grid of the offensive player.,F\nPROMISE-0530,10, When the hit results in sinking a ship  the product shall notify the players that the ship was sunk.,F\nPROMISE-0531,10, When the hit results in sinking a ship  the product shall mark the ship as sunk on the defensive grid of the defensive player.,F\nPROMISE-0532,10, When the hit results in sinking a ship  the product shall indicate the type of ship sunk on the defensive grid of the defensive player.,F\nPROMISE-0533,10, When the hit results in sinking a ship  the product shall mark the ship as sunk on the offensive grid of the offensive player.,F\nPROMISE-0534,10, If the shot was marked as a hit  the product shall allow the offensive player to define a shot.,F\nPROMISE-0535,10, If the shot was marked as a miss  the product shall change the status of the offensive player to defensive,F\nPROMISE-0536,10, If the shot was marked as a miss  the product shall change the status of the defensive player to offensive,F\nPROMISE-0537,10,The product shall continue to assign turns until the game is ended.,F\nPROMISE-0538,10, The product shall end a game when a hit by the offensive player results in sinking the last of the defensive player’s ships  which had not been previously marked as sunk.,F\nPROMISE-0539,10,The product shall allow a player to end a game at any time during the game.,F\nPROMISE-0540,10, When a game is ended  the product shall notify the players that the game is ended.,F\nPROMISE-0541,10, The notification of the game ending will include the reason the game is ended (i.e.  a player has won the game or a player has ended the game).,F\nPROMISE-0542,10, When a game is ended  the product shall allow each player to identify himself or herself as an available player in the list of players.,F\nPROMISE-0543,10, When a game is ended  the product shall allow each player to remove himself or herself from the list of players.,F\nPROMISE-0544,11,The system shall be easy to use by callers and supervisors. Callers and supervisors must be able to accomplish any system task within 2 minutes.,US\nPROMISE-0545,11,The system shall be easy to learn by callers and supervisors. Callers and supervisors shall learn how to use the system within 10 minutes.,US\nPROMISE-0546,11, The system shall be responsive. At the DePaul TeleServices Center  when a supervisor uses the system to view a callers schedule  the system response time must be no more than 2 seconds 90% of the time and no more than 5 seconds the rest of the time.,PE\nPROMISE-0547,11, On a 56k connection  the system response time for all operations must be no more than 6 seconds 90% of the time.,PE\nPROMISE-0548,11, On a 112k connection or faster  the system response time for all operations must be no more than 3 seconds 90% of the time.,PE\nPROMISE-0549,11,The system shall have high availability every day of the year.The system must be available for use between 12:00AM and 6:00PM all days of the year.,A\nPROMISE-0550,11,The system must be available for use between 7:00AM and 11:59PM all days of the year.,A\nPROMISE-0551,11,The product shall be internet browser independent.The product must run using Internet Explorer and Netscape Navigator.,MN\nPROMISE-0552,11,The product shall ensure that only supervisors can advertise empty time slots.The product must ensure that supervisors are allowed to access advertise empty time slots.,SE\nPROMISE-0553,11,The product shall ensure that only supervisors can enter a caller’s schedule.The product must ensure that supervisors are allowed to access advertise empty time slots.,SE\nPROMISE-0554,11,The product shall ensure that only supervisors can view schedule of all callers.The product must ensure that supervisors are allowed to access advertise empty time slots.,SE\nPROMISE-0555,11, The interface shall have a simple look. In the interface and usability survey  90% of beta testers must report that the interface is adequately simple.,LF\nPROMISE-0556,11, The interface shall be appealing to callers and supervisors. In the interface and usability survey  80% of beta testers must report that the interface is appealing.,LF\nPROMISE-0557,12,The product shall provide GUI based Monitoring Services.System Administrators shall monitor message exchanges will predict and determine problems and will also provide internal and external support.,LF\nPROMISE-0558,12,The product shall comply with City of Chicago Application Development Guide Framework.Application Development Framework will provide structured Graphical User Interface.,LF\nPROMISE-0559,12,The product shall provide Monitoring Services.The product shall be easy for System Administrators and DBAs to use after two weeks of usage.,US\nPROMISE-0560,12,The product shall provide Error Tracking Services.The product shall help the System Administrators and DBAs to resolve transaction errors 99% of the time.,US\nPROMISE-0561,12,The product shall provide Portals.The product shall help System Administrators to analyze the transactions with ad-hoc reporting and KPIs (Key Performance Indicators),US\nPROMISE-0562,12,The product shall provide Asynchronous Messaging.The product shall provide asynchronous messaging to enable parallel processing and reduce the overhead in communications,PE\nPROMISE-0563,12,The product shall provide Multi-threading and Load Balancing.The product shall provide multi-threading technology as well as load balancing to overcome data load spikes and traffic.,PE\nPROMISE-0564,12,The product shall process messages within 2 seconds.The product shall have a maximum message processing time of 2 seconds,PE\nPROMISE-0565,12,The product shall poll the system for messages every second.The product shall poll the system for new messages at a maximum of every 1 second.,PE\nPROMISE-0566,12,The product shall provide Dynamic Change Support and Transparent resource addition.The product shall support Transparent Resource addition and Dynamic Change support to provide scalability and avoid service interruptions of more than 1 day.,PE\nPROMISE-0567,12,The product shall be available 99% of the time. Rationale: To avoid service interruption during busiest customer service response periods. The product shall be available 99.99% of the time for regular business days.,A\nPROMISE-0568,12,The product shall be robust.The product shall have fault avoidance based on standards adoption.,FT\nPROMISE-0569,12,The product shall be robust with fault tolerance.The product shall have fault tolerance by using recovery technique compensatory transaction and routing around failures.,FT\nPROMISE-0570,12,The product shall be able to process all transactions.The product shall process minimum of 1 million transactions per year.,SC\nPROMISE-0571,12,The product shall be platform independent.The product shall enable access to any type of development environment and platform.,O\nPROMISE-0572,12,The product shall interface CSR and Datastream.The product shall exchange messages between CSR and EAM on a real time basis without any direct interfaces between two systems.,O\nPROMISE-0573,12, : The product shall continue to operate during upgrade  change or new resource addition.The product shall be able to continue to operate with no interruption in service due to new resource additions.,MN\nPROMISE-0574,12,The product shall be updated on a regular basis.The product will have a new major feature introduced every six months and quarterly maintenance releases every six months.,MN\nPROMISE-0575,12,The product shall operate with every platform.The product shall accommodate any new technology without any change in its architecture.,MN\nPROMISE-0576,12, The product shall have security.The product shall provide authentication  and authorization.,SE\nPROMISE-0577,12,The product shall information Security. The product shall enable privacy by encryption WS-Security and Integrity Checking.,SE\nPROMISE-0578,12,The product shall have audit capabilities.The product shall store messages for a minimum of one year for audit and transaction tracking purposes.,SE\nPROMISE-0579,13,The system shall use IBM and CCR branding standards. The interface must have one IBM logo and one CCR logo.,LF\nPROMISE-0580,13,The system shall be displayable on any work computer a CCR user may have. The system interface shall be formatted to display at 800 by 600 pixel resolution.,LF\nPROMISE-0581,13,The system interface will use CCR nomenclature. The system must use the same nomenclature used in the existing WCS system and reports,LF\nPROMISE-0582,13,The interface must summarize the data in a fashion that’s appealing to CCR management. Every report type must have at least one high level summary shown in the results dashboard/interface,LF\nPROMISE-0583,13,The system must use a point and click interface. 100% of all selection and navigation functionality must be able to be activated/selected by either a mouse or a keyboard,US\nPROMISE-0584,13,The interface shall be easy to use. 90% of users should be able to successfully use the system to generate a report on their first use. The interface should adhere to IBM usability standards.,US\nPROMISE-0585,13,Rreports must be easy to generate. eports must be generated within one button click and exported within one button click (after selecting the report),US\nPROMISE-0586,13,A user shall be able to use the system without instruction after a short amount of time. After one use a user should understand and remember 90% of the system usage and navigation.,US\nPROMISE-0587,13,All CCR users shall receive training before the WCS system is deployed. One training session will be held for 100% of CCR users before system deployment (based on a CCR user being available),US\nPROMISE-0588,13,The WCS system shall use appropriate nomenclature and terminology as defined by the Corporate Community Grants organization. All interfaces and reports will undergo usability tests by CCR users.,US\nPROMISE-0589,13,The WCS system shall be available at all times as long as IBM’s internal intranet W3 is accessible. The WCS system will be available 95% of the time IBM’s internal intranet W3 is available.,A\nPROMISE-0590,13,The WCS system shall have the ability to connect other designated WCS databases currently not included in the system.,SC\nPROMISE-0591,13,The WCS system shall have the ability to include data for regions outside the United States and be accessed by users outside of the United States if approved by CCR management.,SC\nPROMISE-0592,13,The WCS system shall be run in a Windows environment. The WCS shall be accessed by users through the Internet on an IBM computer running a Windows 2000 or above operating system.,O\nPROMISE-0593,13,The Brio portion of the WCS system must be able to export files in spreadsheet form (Microsoft Excel and Lotus 1-2-3 formats).  Brio will provide buttons in the user interface that produce/export reports in .xls or .123 file formats.,O\nPROMISE-0594,13,The WCS system must run queries to the WCS databases once again (on a specified day) to update the available data available to CCR users. Queries must be made to external databases for data refresh once a week.,MN\nPROMISE-0595,13, All changes  updates  and additions to the WCS system shall be updated in the WCS Developer and WCS User Guides. Every change must be reflected in WCS system documentation,MN\nPROMISE-0596,13,The system shall only be accessed by authorized Corporate Community Grants (CCR) employees. Users must be authenticated using their IBM intranet user name and password.,SE\nPROMISE-0597,13,The WCS system shall not allow automatic logins by any user. Cookies containing WCS login information about a user will not be stored on a user’s computer.,SE\nPROMISE-0598,14,Each team owner will need to use a username and password in order to be authenticated to reach the team owner’s page,SE\nPROMISE-0599,14,The Commissioner will be the only authorized user per league to adjust league settings.,SE\nPROMISE-0600,14,Each page shall contain a navigation menu which shows the index into which the current page is contained.,LF\nPROMISE-0601,14,Each page shall contain a similar format with action items in one location and navigation buttons in a different location.,LF\nPROMISE-0602,14,The message board shall contain the 5 most recently posted messages on the league page. The messages will contain up to the first 150 characters of the message.,LF\nPROMISE-0603,14,\tA new user shall be able to navigate through the league and team pages within 30 seconds of reaching the start-up page.,US\nPROMISE-0604,14, \tUsers shall be able to reach the ‘add’  ‘drop’  and ‘trade player’ pages with one click from the team owner’s page.,US\nPROMISE-0605,14,\tUsers shall be able to view at least 25 NFL player statistics and points on each page.,US\nPROMISE-0606,14,The application shall ask the user to confirm dropping a player or trading a player before performing the action.,US\nPROMISE-0607,14,The application shall be able to complete every user transaction within one fourth of a second of receiving the request over the network.,PE\nPROMISE-0608,14,The application shall have a downtime of at most 10 minutes per year.,A\nPROMISE-0609,14,The application shall run on a web application server.,O\nPROMISE-0610,14,A primary database will be used to store cached data for active users.,O\nPROMISE-0611,14,A secondary database will be used to store historical statistics and data.,O\nPROMISE-0612,14,Application updates shall occur between 3AM and 6 AM CST on Wednesday morning during the middle of the NFL season.,MN\nPROMISE-0613,14,New System version releases shall be released at the beginning of each NFL season.,MN\nPROMISE-0614,15, \tThe RFS system should be available 24/7  especially during the budgeting period. The RFS system shall be available 90% of the time all year  and 98% during the budgeting period. 2% of the time  the system will become available within 1 hour of the time that the situation is reported.,A\nPROMISE-0615,15,98% of the user workstations shall meet the minimum hardware requirements prior to implementation. Minimum Hard Disk Space:  400MB.,O\nPROMISE-0616,15,98% of the user workstations shall meet the minimum software requirements prior to implementation: Microsoft Internet Explorer 5.0. Microsoft .NET Framework 1.1. Microsoft Internet Explorer Web Controls,O\nPROMISE-0617,15, \tThe RFS system should be able to easily interface with the BDW environment  in order to load the data. •\t7 data files shall be received from the BDW and loaded into the RFS system within 3 hours one day prior to the start of budgeting process  every year. The data will be as of October month-end. \tTest files (7) shall be loaded up to 5 times throughout the year  from the BDW.,O\nPROMISE-0618,15, \tChanges to revenue and expense formulae must be applied to the RFS system  every year  prior to the budgeting process start. Revenue and expense formulae will be updated within one week of their receipt  after which 2 weeks will be allotted to testing of the budget calculations  to be conducted jointly by the project development team and PCG Finance.,MN\nPROMISE-0619,15, \tNew branches/complexes/districts/regions and revenue/expense accounts added to the BDW  must be reflected in the RFS system. 100% of the entities (mentioned above) available as of October month-end  will be loaded into the RFS system  prior to the budget start process.,MN\nPROMISE-0620,15, \tOnly authorized user shall access the RFS system. ONLY users that exist in the UAS system  with the RFS system assigned to their role(s)  shall access the RFS system and use it.,SE\nPROMISE-0621,15,\tUser access should be limited to the permissions granted to their role(s) Each level in the PCG hierarchy will be assigned a role and users will be assigned to these roles. Access to functionality within RFS system is dependent on the privileges/permission assigned to the role..,SE\nPROMISE-0622,15, \tThe product must comply with the intranet page standards and requirements of ENET Securities. 95% of the product look & feel will be approved by the Architecture group  prior to implementation. The remaining 5% will be corrected and approved within one month of product release.,LF\nPROMISE-0623,15, The intranet pages should display appropriately in all resolutions  from 800x600 and higher. The display of all the pages will be correct in 95% of the resolutions  prior to implementation. The remaining 5% will be corrected within a week of notification of the incorrect display.,LF\nPROMISE-0624,15, \tThe users should be able to easily use the system to successfully complete their budgets  within the stipulated time for completion. With a week of training prior to product implementation  98% of the users shall complete their budgets within the stipulated time.,US\nPROMISE-0625,15, \tThe product interface should be fast. The response time of the product interface should not exceed 2 seconds 90% of the time. For 10% of the time  the response time should not be more than 10 seconds.not be more than 10 seconds.,PE\n', 'data/provenance.json': '{\n  "retrieval_url": "https://raw.githubusercontent.com/AleksandarMitrevski/se-requirements-classification/master/0-datasets/tera-PROMISE%20NFR/nfr.arff",\n  "retrieved": "2026-11-01",\n  "original_notice": "Copyright 2007 Jane Cleland-Huang; CC BY-SA 3.0",\n  "raw_encoding": "cp1252",\n  "raw_sha256": "3d93f83641b76553ddce828c3f80e8c2b70269e51044046ec08b931383dc6c15",\n  "processed_sha256": "1cb11f2a65905cd3eac2672e3e9f6399fc542628a56bf37c0ce734faf843c2b2",\n  "raw_rows": 625,\n  "excluded_rows": [\n    "PROMISE-0426",\n    "PROMISE-0460"\n  ],\n  "reason": "Identical normalized statement labeled SE and F. Both excluded without adjudicating either label.",\n  "processed_rows_before_consistent_dedup": 622,\n  "limitation": "Retrieved from current mirror; byte identity with a pre-April-2020 archived snapshot not independently verified.",\n  "excluded_near_duplicate": "PROMISE-0402",\n  "retained_near_duplicate_representative": "PROMISE-0043",\n  "near_duplicate_audit": "One cross-project pair above 0.90 word-(1,2)-gram TF-IDF cosine; score 0.9563276482856331. Same availability proposition, later-ID representative excluded. Audit was exploratory, after an initial pilot execution; final runs use the cleaned corpus."\n}', 'data/DATA_LICENSE.md': '# Dataset attribution and license\n\nOriginal NFR dataset: Jane Cleland-Huang, Sepideh Mazrouee, Huang Liguo, and Dan Port (2007), PROMISE Software Engineering Repository. The included original artifact states copyright 2007 Jane Cleland-Huang and Creative Commons Attribution-ShareAlike 3.0.\n\nLicense: https://creativecommons.org/licenses/by-sa/3.0/ ; legal code: https://creativecommons.org/licenses/by-sa/3.0/legalcode . Original notices remain in nfr_original.arff. Dataset author endorsement is not implied.\n\nRetrieved from the mirror identified in provenance.json. Original repository starting point: https://openscience.us/repo/requirements/other-requirements/nfr/ . Current mirror availability does not independently prove historical byte identity.\n\nChanges: original CP1252 ARFF converted to UTF-8 CSV; generated stable row IDs; excluded both instances of one statement with conflicting SE/F labels, preserving them in excluded_conflicting_duplicates.csv. These derived dataset files remain under CC BY-SA 3.0. Consistent normalized duplicates are removed by the experiment runner, with retained rows exported in results. Code and project documentation are separate works and are not represented as authored or endorsed by the dataset creators.\n\nThe demo_requirements.csv fixture is synthetic and is not a research dataset or performance benchmark.\n\nAdditional documented change: removed PROMISE-0402, a cross-project near duplicate of PROMISE-0043 (cosine 0.9563 at an audit threshold of 0.90), preserving it in excluded_near_duplicate.csv. Data cleaning was exploratory after an initial pilot; final results must be described accordingly.\n', 'data/excluded_conflicting_duplicates.csv': 'requirement_id,project_id,text,label,normalized\nPROMISE-0426,8,Only registered customers can purchase streaming movies.,SE,only registered customers can purchase streaming movies\nPROMISE-0460,8,Only registered customers can purchase streaming movies.,F,only registered customers can purchase streaming movies\n', 'data/excluded_near_duplicate.csv': 'requirement_id,project_id,text,label\nPROMISE-0402,8, The website shall be available for use 24 hours per day  365 days per year.,A\n'}
for name, content in FILES.items():
    path = BASE / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')
# --no-deps prevents newer transitive versions from entering the project environment.
for req in ['requirements-bootstrap-2020.txt', 'requirements-research-2020.txt']:
    subprocess.run([str(PY), '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', '-r', str(BASE/'config'/req)], check=True, env=ENV)
subprocess.run([str(PY), '-m', 'pip', 'check'], check=True, env=ENV)
subprocess.run([str(PY), '-c', "import sys, sklearn, numpy, scipy, pandas, matplotlib; assert sys.version_info[:3] == (3,7,6); assert sklearn.__version__ == '0.22.2.post1'; print('Historical research environment ready')"], check=True, env=ENV)

import base64
(BASE/"data/nfr_original.arff").write_bytes(base64.b64decode('...'))


## 1. Smoke test (synthetic data only)
This small fixture deliberately contains simple repeated patterns. Its accuracy has no scientific meaning and must not be inserted into the paper. The manifest marks it DEMONSTRATION_ONLY.


In [ ]:
subprocess.run([str(PY), str(BASE/'research/run_experiments.py'), '--data', str(BASE/'data/demo_requirements.csv'), '--output', str(BASE/'results/demo'), '--quick', '--demo'], check=True, env=ENV)


## 2. Select research data
The default uses the bundled reviewed PROMISE CSV. Set `USE_BUNDLED=False` to upload another approved CSV/ARFF. Required CSV columns: requirement_id, project_id, text, label. Keep original NFR label codes; binary conversion occurs inside the runner.

The raw mirrored file contains 625 rows. The supplied reviewed CSV has 622 rows after documented exclusions; the runner additionally removes consistent normalized duplicates. The original raw ARFF is CP1252 encoded. To convert it independently, pass `--encoding cp1252` to convert_promise.py. Do not silently replace decoding errors.

The corpus includes rare subcategories. Inspect class support, missing-training-class warnings and fold-versus-pooled aggregation before interpreting results. Data changes and any further model tuning should be recorded as exploratory.


In [ ]:
RUN_REAL = True
USE_BUNDLED = True
REAL_CSV = BASE/'data/promise_reviewed.csv'
if RUN_REAL and not USE_BUNDLED:
    DATASET_SOURCE = ''
    PROVENANCE_NOTES = ''
    ARFF_ENCODING = 'utf-8-sig'  # Use cp1252 only after checking the source encoding.
    if not DATASET_SOURCE or not PROVENANCE_NOTES:
        raise ValueError('Fill custom dataset provenance before upload.')
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one CSV or ARFF.')
    original_name, payload = next(iter(uploaded.items()))
    ext = pathlib.Path(original_name).suffix.lower()
    raw = BASE/'data'/('custom_input'+ext)
    raw.write_bytes(payload)
    REAL_CSV = BASE/'data/custom_requirements.csv'
    if ext == '.arff':
        subprocess.run([str(PY), str(BASE/'research/convert_promise.py'), str(raw), str(REAL_CSV), '--encoding', ARFF_ENCODING], check=True, env=ENV)
    elif ext == '.csv':
        REAL_CSV.write_bytes(payload)
    else:
        raise ValueError('Only CSV/ARFF research data here; application TXT/DOCX/PDF input is separate.')
    (BASE/'data/custom_provenance.json').write_text(json.dumps({'source':DATASET_SOURCE,'notes':PROVENANCE_NOTES,'sha256':hashlib.sha256(REAL_CSV.read_bytes()).hexdigest()},indent=2))
print('Research data:', REAL_CSV)
print((BASE/'data/provenance.json').read_text() if USE_BUNDLED else 'Custom dataset selected')


## 3. Full binary and exploratory NFR experiments
Ten approaches include baselines, hybrid representations and surface-feature ablations. Nested GroupKFold keeps source projects apart. No GPU is used. The model chosen for deployment is refitted on all development data; the maximum observed CV score is not an independent final test of the winner.

Rare NFR classes can be absent from training or test folds. Inspect the support warnings and report pooled out-of-fold metrics alongside fold means. The fixed-label macro-F1 convention assigns zero to absent test classes. Do not silently remove them to improve the average.


In [ ]:
if RUN_REAL:
    for task in ['binary', 'nfr']:
        subprocess.run([str(PY), str(BASE/'research/run_experiments.py'), '--data', str(REAL_CSV), '--task', task, '--output', str(BASE/'results'/task)], check=True, env=ENV)


## 4. Inspect outputs
`summary.csv` contains outer-fold mean and SD, not a confidence interval. `pooled_metrics.csv` uses pooled held-out predictions. Runtime in `fold_metrics.csv` includes inner search and outer training refit. Review error cases in `oof_predictions.csv`, missing-class columns, source hashes and split IDs before interpreting results.


In [ ]:
try:
    from IPython.display import display, Image
except ImportError:
    def display(value): print(value)
    def Image(filename): return 'Chart saved: ' + filename
for task in (['binary', 'nfr'] if RUN_REAL else ['demo']):
    folder = BASE/'results'/task
    print('TASK:', task, '\n', (folder/'summary.csv').read_text())
    print((folder/'manifest.json').read_text())
    display(Image(filename=str(folder/'comparison.png')))


## 5. Optional untouched external evaluation
Acquire and annotate independent projects before examining final model outputs. Check development/external overlap manually, freeze the model, and evaluate once. The cell remains disabled because no approved external data are supplied. Do not reuse the development dataset as an external test.


In [ ]:
RUN_EXTERNAL = False
EXTERNAL_CSV = BASE/'data/external.csv'
if RUN_EXTERNAL:
    if not RUN_REAL or not EXTERNAL_CSV.exists():
        raise ValueError('Complete real-data training and provide the untouched external CSV first.')
    subprocess.run([str(PY), str(BASE/'research/evaluate_external.py'), '--model', str(BASE/'results/binary/model.joblib'), '--data', str(EXTERNAL_CSV), '--task', 'binary', '--output', str(BASE/'results/external')], check=True, env=ENV)


## 6. Export results
The archive includes code, pinned requirements, data with attribution and license notices, and actual experiment outputs. Derived PROMISE data remain under CC BY-SA 3.0. Preserve those notices when sharing.

Use only real benchmark outputs in the paper. The supplied paper reports an exploratory run; a new execution may differ slightly in timings. No independent external evaluation or user study is supplied. Joblib files are executable-code artifacts; load only trusted files.


In [ ]:
import shutil
EXPORT = BASE/'export'
EXPORT.mkdir(exist_ok=True)
for name in ['results', 'research', 'config', 'data']:
    dest = EXPORT/name
    if dest.exists():
        shutil.rmtree(str(dest))
    shutil.copytree(str(BASE/name), str(dest))
archive = shutil.make_archive(str(BASE/'ReqCheck_research_outputs'), 'zip', str(EXPORT))
print('Archive:', archive)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Local notebook: retrieve the archive from the path above.')
